# V9.3.1 — Partie 2 V6 : Normalisation, DQ, confidence, CarthagoDom et validation simplifiée

Cette V6 repart de la Partie 2 V5.5 mais utilise explicitement les métadonnées de la Partie 1 V9.3.1 : `field_confidence`, `extraction_confidence_score`, recovery, désaccords et contrôles sémantiques.

Objectifs V6 :
- limiter les faux positifs ;
- expliquer clairement pourquoi une valeur est à valider ;
- distinguer **page absente** de **champ présent mais non lu** ;
- afficher l'origine de la cohérence/cross-check dans une cellule synthétique ;
- intégrer une confidence de qualité de donnée 0–100, explicable et non assimilée à une probabilité Qwen ;
- normaliser les montants DOM contenant `:` comme séparateur d'impression ;
- contrôler les couples dates début/fin avec la durée lorsque celle-ci est disponible ;
- conserver les règles V5.5 : CarthagoDom, compte local, modulo 97, numéro DOM, permis RTL, augmentation.


## 1. Imports et configuration


In [ ]:
import hashlib
import json
import math
import re
from collections import defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
EXPECTED_FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'
POSTPROCESS_VERSION = 'DOM_V9_3_1_PART2_DQ_CONFIDENCE_V6'
NORMALIZATION_VERSION = 'NORM_V9_3_1_V6_CONFIDENCE_AWARE'
MAX_DOSSIERS = None   # mettre 10 pour un test

OUTPUT_ROOT = Path('/mnt/data/domiciliations_v9')
RAW_JSON_DIR = OUTPUT_ROOT / '01_extraction_raw' / 'json_dossiers'
POST_ROOT = OUTPUT_ROOT / '02_postprocessing'
PROCESSED_JSON_DIR = POST_ROOT / 'processed_json'
VALIDATION_XLSX = POST_ROOT / 'validation_domiciliations_v9_3_1_v6.xlsx'
DOSSIERS_CSV = POST_ROOT / 'dossiers_a_valider_v6.csv'
FIELDS_CSV = POST_ROOT / 'champs_detail_v6.csv'
ANOMALIES_CSV = POST_ROOT / 'anomalies_v6.csv'
RETRY_CSV = POST_ROOT / 'vlm_retry_requests_v6.csv'
DQ_SUMMARY_CSV = POST_ROOT / 'dq_summary_v6.csv'
FIELD_CONFIDENCE_CSV = POST_ROOT / 'field_confidence_v6.csv'

# Référentiel CarthagoDom — V5
# Le fichier peut être placé dans /mnt/data ou le chemin peut être modifié ici.
CARTHAGO_XLSX = Path('/mnt/data/Dossier de domiciliation.xlsx')
CARTHAGO_CLIENT_COL = 'Client'
CARTHAGO_NAME_COL = 'Nom complet/Raison sociale'
# Laisser None pour auto-détection prudente ; renseigner le nom exact si nécessaire.
CARTHAGO_DOM_REF_COL = None
CARTHAGO_TYPE_COL = None

CARTHAGO_MATCH_CSV = POST_ROOT / 'carthago_domiciliation_match_v6.csv'

POST_ROOT.mkdir(parents=True,exist_ok=True)
PROCESSED_JSON_DIR.mkdir(parents=True,exist_ok=True)

print('RAW :',RAW_JSON_DIR)
print('Post:',POST_ROOT)


EXPECTED_SOURCE_PIPELINE_VERSION = 'DOM_V9_3_1_PART1_LAYOUT_RECOVERY_CONFIDENCE'
EXPECTED_CORE_DOC_TYPES = {
    'ENGAGEMENT_DOMICILIATION',
    'CONTRAT_TRAVAIL',
    'CONTRAT_SPECIFIQUE',
    'TITRE_TRAVAIL',
}
# Une page attendue absente est signalée en REVIEW, pas BLOQUANT automatiquement,
# afin d'éviter un faux positif si le dossier métier est exceptionnel.
MISSING_EXPECTED_PAGE_SEVERITY = 'REVIEW'

# Confidence V6 : seuils de qualification, pas seuils automatiques de rejet.
CONFIDENCE_HIGH = 90
CONFIDENCE_MEDIUM = 70
CONFIDENCE_CRITICAL_REVIEW = 70


## 2. Schéma — mêmes 99 champs V8.1 / V9


In [ ]:
FIELD_SCHEMA = {'ENGAGEMENT_DOMICILIATION': ['DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL', 'DOM_ADRESSE_CLIENT', 'DOM_AGENCE_DOMICILIATAIRE', 'DOM_NUMERO_CONTRAT', 'DOM_DUREE_CONTRAT_MOIS', 'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR', 'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_TAUX_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE', 'DOM_DATE_SIGNATURE'], 'CONTRAT_TRAVAIL': ['CTR_REFERENCE_DOCUMENT', 'CTR_TYPE', 'CTR_EMPLOYEUR', 'CTR_ACTIVITE_EMPLOYEUR', 'CTR_DUREE_MOIS', 'CTR_DATE_DEBUT_CONTRAT', 'CTR_POSTE', 'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_PERE_NOM_PRENOM', 'CTR_MERE_NOM_PRENOM', 'CTR_NATIONALITE', 'CTR_DATE_NAISSANCE', 'CTR_LIEU_PAYS_NAISSANCE', 'CTR_ADRESSE_ALGERIE', 'CTR_QUALIFICATION', 'CTR_NUMERO_PERMIS_TRAVAIL', 'CTR_DATE_DELIVRANCE_PERMIS', 'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_SALAIRE_BRUT', 'CTR_SALAIRE_NET', 'CTR_AFFILIATION_SS', 'CTR_NUMERO_EMPLOYEUR', 'CTR_DATE_SIGNATURE', 'CTR_REFERENCE_DOMICILIATION', 'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTR_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTR_CACHET_EMPLOYEUR_PRESENT'], 'CONTRAT_SPECIFIQUE': ['CTS_REFERENCE_DOCUMENT', 'CTS_SAP_ID', 'CTS_EMPLOYEUR', 'CTS_ACTIVITE_EMPLOYEUR', 'CTS_DUREE_MOIS', 'CTS_DATE_DEBUT_CONTRAT', 'CTS_POSTE', 'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_PERE_NOM_PRENOM', 'CTS_MERE_NOM_PRENOM', 'CTS_NATIONALITE', 'CTS_DATE_NAISSANCE', 'CTS_LIEU_PAYS_NAISSANCE', 'CTS_ADRESSE_ALGERIE', 'CTS_QUALIFICATION', 'CTS_NUMERO_PERMIS_TRAVAIL', 'CTS_DATE_DELIVRANCE_PERMIS', 'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'CTS_LIGNE_SALAIRE_BRUTE', 'CTS_SALAIRE_NET', 'CTS_SALAIRE_NET_ANCIEN', 'CTS_MENTION_AU_LIEU_DE_PRESENTE', 'CTS_PART_TRANSFERABLE', 'CTS_PART_PAYABLE_DZD', 'CTS_NUMERO_SS_PAYS_ORIGINE', 'CTS_NUMERO_SS_ALGERIE', 'CTS_DATE_DOCUMENT', 'CTS_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTS_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTS_CACHET_EMPLOYEUR_PRESENT', 'CTS_VISA_INSPECTION_TRAVAIL_PRESENT'], 'TITRE_TRAVAIL': ['TTR_NUMERO_PERMIS', 'TTR_NUMERO_MANUSCRIT', 'TTR_POSTE', 'TTR_DUREE', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR', 'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A', 'TTR_DATE_DELIVRANCE', 'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE', 'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE', 'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT'], 'PERMIS_TRAVAIL_COUVERTURE': ['PTR_NUMERO_SERIE', 'PTR_WILAYA', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT']}

ALL_FIELDS=[f for fields in FIELD_SCHEMA.values() for f in fields]
assert len(ALL_FIELDS)==99 and len(set(ALL_FIELDS))==99
schema_hash=hashlib.sha256(json.dumps(FIELD_SCHEMA,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
assert schema_hash==EXPECTED_FIELD_SCHEMA_HASH
print('✅ Schéma 99 champs vérifié |',schema_hash[:16]+'…')


## 3. Typage des champs — configuration évolutive


In [ ]:
AMOUNT_FIELDS={
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE','DOM_MONTANT_TOTAL_DOMICILIE',
    'CTR_SALAIRE_BRUT','CTR_SALAIRE_NET','CTS_SALAIRE_NET','CTS_SALAIRE_NET_ANCIEN',
    'CTS_PART_TRANSFERABLE','CTS_PART_PAYABLE_DZD',
}
PERCENT_FIELDS={'DOM_TAUX_TRANSFERABLE'}
INTEGER_FIELDS={'DOM_DUREE_CONTRAT_MOIS','CTR_DUREE_MOIS','CTS_DUREE_MOIS'}
BOOLEAN_FIELDS={
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE','CTR_SIGNATURE_EMPLOYEUR_PRESENTE','CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE','CTS_SIGNATURE_TRAVAILLEUR_PRESENTE','CTS_SIGNATURE_EMPLOYEUR_PRESENTE',
    'CTS_CACHET_EMPLOYEUR_PRESENT','CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE','TTR_CACHET_PRESENT','PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
DATE_FIELDS={f for f in ALL_FIELDS if '_DATE_' in f or f.startswith('TTR_DATE_') or f in {'DOM_DATE_SIGNATURE','CTS_DATE_DOCUMENT'}}
REFERENCE_FIELDS={
    'DOM_COMPTE_LOCAL','DOM_NUMERO_CONTRAT','CTR_REFERENCE_DOCUMENT','CTR_NUMERO_PERMIS_TRAVAIL',
    'CTR_REFERENCE_DOMICILIATION','CTS_REFERENCE_DOCUMENT','CTS_SAP_ID','CTS_NUMERO_PERMIS_TRAVAIL',
    'TTR_NUMERO_PERMIS','TTR_NUMERO_MANUSCRIT','PTR_NUMERO_SERIE',
}

FIELD_TYPES={}
for f in ALL_FIELDS:
    if f in AMOUNT_FIELDS: FIELD_TYPES[f]='amount'
    elif f in PERCENT_FIELDS: FIELD_TYPES[f]='percentage'
    elif f in INTEGER_FIELDS: FIELD_TYPES[f]='integer'
    elif f in BOOLEAN_FIELDS: FIELD_TYPES[f]='boolean'
    elif f in DATE_FIELDS: FIELD_TYPES[f]='date'
    elif f in REFERENCE_FIELDS: FIELD_TYPES[f]='reference'
    else: FIELD_TYPES[f]='text'

print(pd.Series(FIELD_TYPES).value_counts())


## 4. Normalisation traçable


In [ ]:
NULL_TEXTS={'','NULL','NONE','N/A','NA','NÉANT','NEANT','ILLISIBLE','NON LISIBLE'}

def result(raw,normalized,status,rule,message=None):
    return {'raw':raw,'normalized':normalized,'status':status,'rule':rule,
            'changed':(normalized != raw),'message':message}


def normalize_amount_trace(raw):
    """
    Normalisation conservatrice des montants.
    Principe réglementaire :
      - ne jamais modifier/inventer un chiffre ;
      - AUTO_OK uniquement si la structure des séparateurs est déterministe ;
      - REVIEW dès qu'une interprétation économique reste plausible.
    """
    if raw is None:
        return result(raw,None,'MISSING','AMOUNT_NULL')

    if isinstance(raw,(int,float,Decimal)) and not isinstance(raw,bool):
        try:
            return result(raw,round(float(raw),2),'RAW_OK','AMOUNT_NUMERIC')
        except Exception:
            pass

    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS:
        return result(raw,None,'MISSING','AMOUNT_NULL_TEXT')

    # Retirer seulement les libellés de devise connus.
    t=s.upper().replace('DZD','').replace('DA','').replace('EUR','').replace('€','').strip()
    neg=t.startswith('-')
    t=t.lstrip('+-').strip()

    # Caractères non autorisés : aucune correction OCR de type O->0 / B->8.
    if not re.fullmatch(r"[0-9., '\u2019]+", t or ''):
        return result(raw,None,'REVIEW','AMOUNT_NON_NUMERIC',
                      'caractère non numérique ambigu')

    # Les séparateurs espace/apostrophe sont acceptés comme milliers
    # seulement si leurs groupes sont structurellement cohérents.
    if re.search(r"[ '\u2019]", t):
        chunks=[x for x in re.split(r"[ '\u2019]+",t) if x]
        # Cas "454 835,67" / "454 835.67" : groupes milliers + décimales explicites.
        if len(chunks) >= 2:
            last=chunks[-1]
            # Si le dernier bloc isolé contient 1 ou 2 chiffres sans . ou ,
            # ex. "454 835 67", on refuse de deviner qu'il s'agit de décimales.
            if re.fullmatch(r'\d{1,2}', last):
                return result(raw,None,'REVIEW','AMOUNT_SPACE_DECIMAL_AMBIGUOUS',
                              'dernier groupe espace/apostrophe ambigu')
            # Tous les groupes intermédiaires doivent être des milliers.
            for ch in chunks[1:-1]:
                if not re.fullmatch(r'\d{3}', ch):
                    return result(raw,None,'REVIEW','AMOUNT_SPACE_GROUPING_AMBIGUOUS')
        t=re.sub(r"[ '\u2019]",'',t)

    if not re.fullmatch(r'[0-9.,]+',t or ''):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')

    dots=t.count('.')
    commas=t.count(',')
    dec_sep=None
    rule=''

    if dots and commas:
        last_dot=t.rfind('.')
        last_comma=t.rfind(',')
        candidate='.' if last_dot>last_comma else ','
        tail=t.split(candidate)[-1]
        if len(tail)==2:
            dec_sep=candidate
            rule='AMOUNT_MIXED_LAST_2_DECIMALS'
        elif len(tail)==1:
            dec_sep=candidate
            rule='AMOUNT_MIXED_LAST_1_DECIMAL_PAD_ZERO'
        else:
            return result(raw,None,'REVIEW','AMOUNT_MIXED_AMBIGUOUS')

    elif dots>1 or commas>1:
        sep='.' if dots else ','
        groups=t.split(sep)
        tail=groups[-1]

        # Exemple validé : 454.835.67 -> 454835.67
        if len(tail)==2 and all(g.isdigit() for g in groups):
            dec_sep=sep
            rule='AMOUNT_MULTI_GROUP_FINAL_2_DECIMALS'
        elif len(tail)==1 and all(g.isdigit() for g in groups) and all(len(g)==3 for g in groups[1:-1]):
            dec_sep=sep
            rule='AMOUNT_MULTI_GROUP_FINAL_1_DECIMAL_PAD_ZERO'
        elif all(len(g)==3 for g in groups[1:]):
            dec_sep=None
            rule='AMOUNT_MULTI_THOUSANDS'
        else:
            return result(raw,None,'REVIEW','AMOUNT_MULTI_AMBIGUOUS')

    elif dots==1 or commas==1:
        sep='.' if dots else ','
        left,right=t.split(sep)
        if len(right)==2:
            dec_sep=sep
            rule='AMOUNT_SINGLE_2_DECIMALS'
        elif len(right)==1:
            # Le séparateur décimal est explicite : 47719857,6 = 47719857.60.
            # Aucun chiffre économique n'est inventé ; le zéro final est
            # uniquement une représentation à 2 décimales.
            dec_sep=sep
            rule='AMOUNT_SINGLE_1_DECIMAL_PAD_ZERO'
        elif len(right)==3:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_3DIGITS_AMBIGUOUS')
        else:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_AMBIGUOUS')
    else:
        rule='AMOUNT_INTEGER'

    if dec_sep:
        pos=t.rfind(dec_sep)
        int_part=re.sub(r'[.,]','',t[:pos])
        dec=t[pos+1:]
        canonical=int_part+'.'+dec
    else:
        canonical=re.sub(r'[.,]','',t)

    if neg:
        canonical='-'+canonical

    try:
        val=round(float(Decimal(canonical)),2)
    except (InvalidOperation,ValueError):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')

    return result(raw,val,'AUTO_OK' if str(raw)!=str(val) else 'RAW_OK',rule)


DATE_FORMATS=['%d/%m/%Y','%d-%m-%Y','%d.%m.%Y','%Y-%m-%d','%Y/%m/%d','%Y.%m.%d']
def normalize_date_trace(raw):
    if raw is None: return result(raw,None,'MISSING','DATE_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','DATE_NULL_TEXT')
    s=re.sub(r'\s+','',s)
    for fmt in DATE_FORMATS:
        try:
            dt=datetime.strptime(s,fmt)
            out=dt.strftime('%d/%m/%Y')
            return result(raw,out,'RAW_OK' if s==out else 'AUTO_OK','DATE_'+fmt.replace('%',''))
        except ValueError:
            pass
    return result(raw,None,'REVIEW','DATE_UNPARSEABLE')


def normalize_integer_trace(raw):
    if raw is None: return result(raw,None,'MISSING','INTEGER_NULL')
    if isinstance(raw,int) and not isinstance(raw,bool): return result(raw,raw,'RAW_OK','INTEGER_NUMERIC')
    s=str(raw).strip(); m=re.fullmatch(r'\s*(\d+)\s*(?:mois)?\s*',s,flags=re.I)
    if not m: return result(raw,None,'REVIEW','INTEGER_AMBIGUOUS')
    v=int(m.group(1)); return result(raw,v,'RAW_OK' if str(v)==s else 'AUTO_OK','INTEGER_EXTRACT')


def normalize_percentage_trace(raw):
    if raw is None: return result(raw,None,'MISSING','PERCENT_NULL')
    s=str(raw).strip().replace('\u00a0',' ')
    has_pct='%' in s
    s=s.replace('%','').replace(' ','').replace(',','.')
    if not re.fullmatch(r'[+-]?\d+(?:\.\d+)?',s): return result(raw,None,'REVIEW','PERCENT_AMBIGUOUS')
    v=float(s)
    if not has_pct and 0 < v <= 1:
        v*=100; rule='PERCENT_FRACTION_TO_PERCENT'
    else: rule='PERCENT_DIRECT'
    if not (0 <= v <= 100): return result(raw,None,'REVIEW','PERCENT_OUT_OF_RANGE')
    v=round(v,2); return result(raw,v,'AUTO_OK' if str(raw).strip()!=str(v) else 'RAW_OK',rule)


def normalize_boolean_trace(raw):
    if raw is None: return result(raw,None,'MISSING','BOOL_NULL')
    if isinstance(raw,bool): return result(raw,raw,'RAW_OK','BOOL_NATIVE')
    s=str(raw).strip().upper()
    if s in {'TRUE','VRAI','OUI','YES','1'}: return result(raw,True,'AUTO_OK','BOOL_TRUE_TEXT')
    if s in {'FALSE','FAUX','NON','NO','0'}: return result(raw,False,'AUTO_OK','BOOL_FALSE_TEXT')
    return result(raw,None,'REVIEW','BOOL_AMBIGUOUS')



# ---------------------------------------------------------------------
# V5.2 — Normalisation métier de CTR_REFERENCE_DOMICILIATION
# Format structuré canonique : PREFIXE|AAAA.T|40|SEQUENCE|DZD
# Exemple : 271901|2026.1|40|001345|DZD
#
# Une seconde représentation, dédiée au rapprochement Carthago, est
# construite par normalize_domiciliation_compare_key() :
#   271901|2026.1|40|001345|DZD -> 2719012026140001345DZD
# ---------------------------------------------------------------------
PIPE_TRANSLATION = str.maketrans({
    '¦': '|',
    '│': '|',
    '｜': '|',
})


def normalize_domiciliation_compare_key(raw):
    """
    Clé de comparaison Carthago / NUMERO_DOMICILIATION_VALIDE.

    Règle métier demandée :
      - supprimer tous les caractères et toutes les lettres ;
      - conserver uniquement les chiffres, dans leur ordre d'origine ;
      - ajouter DZD à la fin.

    Aucune conversion OCR de lettre vers chiffre (O->0, I->1, etc.).
    Ainsi une erreur OCR ne peut pas être silencieusement inventée.
    """
    if raw is None:
        return None
    s = str(raw).replace('\u00a0', ' ').strip()
    if not s or s.upper() in NULL_TEXTS:
        return None
    digits = re.sub(r'\D', '', s)
    if not digits:
        return None
    return f'{digits}DZD'


def normalize_ctr_reference_domiciliation_trace(raw):
    if raw is None:
        return result(raw, None, 'MISSING', 'CTR_DOM_REF_NULL')

    s = str(raw).replace('\u00a0', ' ').strip()
    if s.upper() in NULL_TEXTS:
        return result(raw, None, 'MISSING', 'CTR_DOM_REF_NULL_TEXT')

    # Variantes typographiques sûres du séparateur vertical.
    t = s.translate(PIPE_TRANSLATION)
    t = re.sub(r'\s*\|\s*', '|', t)
    parts = [x.strip() for x in t.split('|')]

    # Si la devise entière est absente et qu'il n'y a que 4 blocs,
    # on ajoute un bloc vide ; pour un transfert salaire la devise métier
    # attendue est DZD. Le cas avec pipe final produit déjà 5 blocs.
    if len(parts) == 4:
        parts.append('')

    if len(parts) != 5:
        return result(
            raw, None, 'REVIEW', 'CTR_DOM_REF_STRUCTURE_EXPECTED_5_BLOCKS',
            'format attendu PREFIXE|AAAA.T|40|SEQUENCE|DZD'
        )

    prefix_raw, periode_raw, operation, sequence, devise_raw = parts

    # Préfixe : autoriser les séparateurs dans 27-19-01, mais jamais
    # convertir une lettre OCR en chiffre.
    prefix = re.sub(r'[^0-9]', '', prefix_raw)
    if not re.fullmatch(r'\d{6}', prefix):
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_PREFIX_INVALID',
                      'préfixe attendu sur 6 chiffres; ex. 27-19-01 -> 271901')

    # Période acceptée :
    #   2026.1 / 2026.T1 / 2026.01 / 2026.T01
    #   20261 / 2026T1
    # Canonique : 2026.1
    periode = re.sub(r'\s+', '', periode_raw.upper())
    patterns = [
        r'^(\d{4})\.(?:T)?0?([1-4])$',
        r'^(\d{4})T0?([1-4])$',
        r'^(\d{4})([1-4])$',
    ]
    m = None
    for pat in patterns:
        m = re.fullmatch(pat, periode)
        if m:
            break
    if not m:
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_YEAR_QUARTER_INVALID',
                      'période attendue AAAA.1..4, AAAA.T1..T4, AAAAT1..T4 ou AAAA1..4')
    year, quarter = m.group(1), m.group(2)

    # Transferts de salaire : code opération 40. Pas de correction 4O -> 40.
    if operation != '40':
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_OPERATION_NOT_40',
                      'code transfert salaire attendu = 40')

    # Séquence conservée comme texte pour garder les zéros initiaux.
    if not re.fullmatch(r'\d+', sequence):
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_SEQUENCE_INVALID',
                      'séquence non numérique; zéros initiaux à conserver')

    # Pour les transferts salaire Algérie -> étranger, une devise absente
    # ou tronquée D / DZ est complétée en DZD. Toute autre devise reste REVIEW.
    devise_letters = re.sub(r'[^A-Z]', '', devise_raw.upper())
    if devise_letters in {'', 'D', 'DZ', 'DZD'}:
        devise = 'DZD'
    else:
        return result(raw, None, 'REVIEW', 'CTR_DOM_REF_CURRENCY_NOT_DZD',
                      'devise attendue DZD; seules les formes vide/D/DZ/DZD sont auto-complétées')

    canonical = f'{prefix}|{year}.{int(quarter)}|40|{sequence}|DZD'
    status = 'RAW_OK' if canonical == s else 'AUTO_OK'
    out = result(raw, canonical, status, 'CTR_DOM_REF_STRUCTURED_CANONICAL_V5_1')
    out['components'] = {
        'prefix': prefix,
        'year': int(year),
        'quarter': int(quarter),
        'operation': '40',
        'sequence': sequence,
        'currency': 'DZD',
    }
    out['compare_key'] = normalize_domiciliation_compare_key(canonical)
    return out

def normalize_reference_trace(raw):
    if raw is None: return result(raw,None,'MISSING','REF_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','REF_NULL_TEXT')
    # Conservateur : espaces périphériques et autour de / - uniquement.
    out=re.sub(r'\s*([/\-])\s*',r'\1',re.sub(r'\s+',' ',s)).strip()
    return result(raw,out,'AUTO_OK' if out!=s else 'RAW_OK','REF_SPACING_ONLY')


def normalize_text_trace(raw):
    if raw is None: return result(raw,None,'MISSING','TEXT_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','TEXT_NULL_TEXT')
    # Pas de correction orthographique / casse.
    return result(raw,s,'AUTO_OK' if s!=raw else 'RAW_OK','TEXT_TRIM_ONLY')


def normalize_field_trace(field,raw):
    if field == 'CTR_REFERENCE_DOMICILIATION':
        return normalize_ctr_reference_domiciliation_trace(raw)
    typ=FIELD_TYPES.get(field,'text')
    return {
        'amount':normalize_amount_trace,
        'date':normalize_date_trace,
        'integer':normalize_integer_trace,
        'percentage':normalize_percentage_trace,
        'boolean':normalize_boolean_trace,
        'reference':normalize_reference_trace,
        'text':normalize_text_trace,
    }[typ](raw)

# Tests de non-régression des montants.
assert normalize_amount_trace('454.835.67')['normalized']==454835.67
assert normalize_amount_trace('23.340.43')['normalized']==23340.43
assert normalize_amount_trace('23,340.43')['normalized']==23340.43
assert normalize_amount_trace('23.340,43')['normalized']==23340.43
assert normalize_amount_trace('23 340,43')['normalized']==23340.43
assert normalize_amount_trace('23,340')['status']=='REVIEW'
assert normalize_amount_trace('454 835 67')['status']=='REVIEW'
assert normalize_amount_trace('454.835.6')['normalized']==454835.60
assert normalize_amount_trace('47 719 857,6')['normalized']==47719857.60
assert normalize_amount_trace('47.719.857,6')['normalized']==47719857.60
assert normalize_amount_trace('454 835 67')['status']=='REVIEW'
assert normalize_amount_trace('454.83O.67')['status']=='REVIEW'
print('✅ Montants testés : 454.835.67 -> 454835.67 | 47 719 857,6 -> 47719857.60')


# Tests V5.2 — référence domiciliation.
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|DZD')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901 | 2026.T1 | 40 | 001345 | dzd')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('27-19-01|2026.1|40|001345|DZD')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|DZ')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|D')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('27-19-01|20261|40|001345|DZD')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('27-19-01|2026T1|40|001345|DZD')['normalized'] == '271901|2026.1|40|001345|DZD'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.5|40|001345|DZD')['status'] == 'REVIEW'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|4O|001345|DZD')['status'] == 'REVIEW'
assert normalize_ctr_reference_domiciliation_trace('271901|2026.1|40|001345|EUR')['status'] == 'REVIEW'
assert normalize_domiciliation_compare_key('271901|2026.1|40|001345|DZD') == '2719012026140001345DZD'
assert normalize_domiciliation_compare_key('2719012026140001345DZD') == '2719012026140001345DZD'
print('✅ CTR_REFERENCE_DOMICILIATION : normalisation structurée V5.2 active')
print('✅ NUMERO_DOMICILIATION_VALIDE : clé compacte chiffres + DZD active')


## 5. Contrôles croisés — sans appel Qwen


In [ ]:
CROSS_DOCUMENT_GROUPS=[
    {'name':'SALAIRE_NET','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_SALAIRE_NET_MENSUEL','CONTRAT_TRAVAIL':'CTR_SALAIRE_NET','CONTRAT_SPECIFIQUE':'CTS_SALAIRE_NET'}},
    {'name':'PART_TRANSFERABLE','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_PART_TRANSFERABLE','CONTRAT_SPECIFIQUE':'CTS_PART_TRANSFERABLE'}},
    {'name':'DATE_DEBUT_CONTRAT','kind':'date','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_DATE_DEBUT_CONTRAT','CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_CONTRAT','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_CONTRAT'}},
    {'name':'NUMERO_PERMIS','kind':'reference','fields':{
        'CONTRAT_TRAVAIL':'CTR_NUMERO_PERMIS_TRAVAIL','CONTRAT_SPECIFIQUE':'CTS_NUMERO_PERMIS_TRAVAIL','TITRE_TRAVAIL':'TTR_NUMERO_PERMIS'}},
    {'name':'DATE_DEBUT_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_DEBUT'}},
    {'name':'DATE_FIN_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_FIN_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_FIN_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_FIN'}},
]

def comparable(v): return v not in (None,'')
def same_value(kind,a,b):
    if not comparable(a) or not comparable(b): return True
    if kind=='amount': return abs(float(a)-float(b))<=0.01
    return str(a)==str(b)


In [ ]:

# =====================================================================
# FIELD_CONFIDENCE + DQ_SCORE + APPLICABILITE METIER — V3
# =====================================================================
# IMPORTANT :
# - NORMALIZATION_STATUS décrit UNIQUEMENT la normalisation d'une cellule.
# - DQ_STATUS_FIELD décrit la fiabilité métier du champ après tous les contrôles.
# - VALIDATION_AUTO décrit le statut GLOBAL du dossier.
#
# Un champ peut donc être NORMALIZATION_STATUS=AUTO_OK mais
# DQ_STATUS_FIELD=BLOCKED si sa valeur est en conflit avec un autre document.
#
# Les scores sont des INDICES INTERNES EXPLICABLES, pas des probabilités Qwen.

CRITICAL_FIELDS = {
    'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT',
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE',
    'CTR_NUMERO_PERMIS_TRAVAIL','CTR_DATE_DEBUT_VALIDITE_PERMIS',
    'CTR_DATE_FIN_VALIDITE_PERMIS','CTR_SALAIRE_NET',
    'CTS_NUMERO_PERMIS_TRAVAIL','CTS_DATE_DEBUT_VALIDITE_PERMIS',
    'CTS_DATE_FIN_VALIDITE_PERMIS','CTS_SALAIRE_NET',
    'TTR_NUMERO_PERMIS','TTR_DATE_DEBUT','TTR_DATE_FIN',
    'TTR_NOM','TTR_PRENOM'
}

ALWAYS_INFO_ONLY_FIELDS = {
    'CTS_SAP_ID',
    'TTR_NUMERO_MANUSCRIT',
    'PTR_NUMERO_SERIE',
    'PTR_WILAYA',
    'PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
    'CTS_NUMERO_SS_PAYS_ORIGINE',
    'CTS_NUMERO_SS_ALGERIE',
}

NON_PENALIZING_QUALITY_FLAGS = {
    'MIGRATED_FROM_V8_1',
    'CRITICAL_MISSING_OR_STRUCTURALLY_SUSPECT',
    'LOW_FILL_RATE',
    'FIELD_REREAD_BY_HD_RETRY',
}

DQ_WEIGHTS = {
    'critical_completeness': 25,
    'cross_document': 30,
    'normalization': 15,
    'business_validity': 15,
    'extraction_quality': 10,
    'classification_quality': 5,
}
assert sum(DQ_WEIGHTS.values()) == 100

SEVERITY_RANK = {'OK':0, 'INFO':0, 'REVIEW':1, 'BLOQUANT':2}

DATE_ORDER_FIELDS = {
    'CONTRAT': {'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT'},
    'PERMIS_CTR': {'CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS'},
    'PERMIS_CTS': {'CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS'},
    'PERMIS_TTR': {'TTR_DATE_DEBUT','TTR_DATE_FIN'},
}

CROSSCHECK_FIELD_MAP = {
    g['name']: set(g['fields'].values())
    for g in CROSS_DOCUMENT_GROUPS
}


def crosscheck_group_applicable(group, business_context):
    """Désactive les comparaisons dont l'écart est attendu en augmentation."""
    typ=(business_context or {}).get('TYPE_DOSSIER')
    if typ=='AUGMENTATION' and group.get('name') in {
        'SALAIRE_NET','PART_TRANSFERABLE','DATE_DEBUT_CONTRAT'
    }:
        return False
    return True


def _meaningful(v):
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip().upper() not in NULL_TEXTS
    return True


def _first_value(records, field):
    for r in records:
        v=(r.get('normalized_data') or {}).get(field)
        if comparable(v):
            return v
    return None


def derive_business_context(processed_records):
    cts_records=[r for r in processed_records if r.get('doc_type')=='CONTRAT_SPECIFIQUE']
    has_cts=bool(cts_records)

    old_salary=None
    mention_au_lieu=False
    cts_date_document=None

    for r in cts_records:
        nd=r.get('normalized_data') or {}
        if old_salary is None and comparable(nd.get('CTS_SALAIRE_NET_ANCIEN')):
            old_salary=nd.get('CTS_SALAIRE_NET_ANCIEN')
        if nd.get('CTS_MENTION_AU_LIEU_DE_PRESENTE') is True:
            mention_au_lieu=True
        if cts_date_document is None and comparable(nd.get('CTS_DATE_DOCUMENT')):
            cts_date_document=nd.get('CTS_DATE_DOCUMENT')

    evidence=[]
    if comparable(old_salary):
        evidence.append('CTS_SALAIRE_NET_ANCIEN_PRESENT')
    if mention_au_lieu:
        evidence.append('CTS_MENTION_AU_LIEU_DE_PRESENTE_TRUE')

    if evidence:
        type_dossier='AUGMENTATION'
    elif has_cts:
        type_dossier='NOUVEAU_CONTRAT'
        evidence.append('CTS_SANS_INDICATEUR_AUGMENTATION')
    else:
        type_dossier='A_DETERMINER'
        evidence.append('ABSENCE_CONTRAT_SPECIFIQUE')

    return {
        'TYPE_DOSSIER':type_dossier,
        'TYPE_DOSSIER_MOTIF':' | '.join(evidence),
        # V5 : la date du document CTS reste une information documentaire.
        # Elle NE représente PAS la date d'effet métier de l'augmentation.
        'CTS_DATE_DOCUMENT_AUGMENTATION':cts_date_document if type_dossier=='AUGMENTATION' else None,
        'CTS_DATE_AUGMENTATION':None,
        'PERIODE_EFFET_AUGMENTATION':None,
    }


def field_policy(field, business_context):
    typ=(business_context or {}).get('TYPE_DOSSIER')

    if field in ALWAYS_INFO_ONLY_FIELDS:
        return 'INFO_ONLY','INFO_ONLY'

    if field=='CTS_SALAIRE_NET_ANCIEN' and typ=='NOUVEAU_CONTRAT':
        return 'NON_APPLICABLE','INFO_ONLY'

    if field=='DOM_MONTANT_TOTAL_DOMICILIE' and typ=='AUGMENTATION':
        return 'NON_APPLICABLE','INFO_ONLY'

    if field in CRITICAL_FIELDS:
        return 'APPLICABLE','CRITICAL'

    return 'APPLICABLE','STANDARD'


def is_dq_relevant_field(field, business_context):
    return field_policy(field,business_context)[1] != 'INFO_ONLY'


def is_info_only_document(doc_type, business_context):
    fields=FIELD_SCHEMA.get(doc_type,[])
    return bool(fields) and all(
        not is_dq_relevant_field(f,business_context)
        for f in fields
    )


def relevant_critical_missing(rec, business_context):
    """Champs critiques réellement absents après normalisation Partie 2."""
    dt=rec.get('doc_type')
    nd=rec.get('normalized_data') or {}
    tr=rec.get('normalization_trace') or {}
    raw=rec.get('raw_data') or {}
    missing=[]
    for field in FIELD_SCHEMA.get(dt,[]):
        if field not in CRITICAL_FIELDS or not is_dq_relevant_field(field,business_context):
            continue
        if comparable(nd.get(field)):
            continue
        # RAW présent mais ambigu => FORMAT_REVIEW, pas MISSING.
        if _meaningful(raw.get(field)) and (tr.get(field) or {}).get('status')=='REVIEW':
            continue
        if not _meaningful(raw.get(field)):
            missing.append(field)
    return sorted(set(missing))


def penalizing_quality_flags(rec):
    return [
        str(x) for x in (rec.get('quality_flags') or [])
        if str(x) not in NON_PENALIZING_QUALITY_FLAGS
    ]


def parse_date_safe(v):
    if not v:
        return None
    try:
        return datetime.strptime(str(v), '%d/%m/%Y')
    except Exception:
        return None


def page_regulatory_flags(rec, business_context=None):
    """Recalcule les anomalies à partir de l'état courant Partie 2."""
    flags=[]
    info_doc=is_info_only_document(rec.get('doc_type'),business_context)
    if rec.get('classification_review_required') is True and not info_doc:
        flags.append('CLASSIFICATION_REVIEW_REQUIRED')
    if not info_doc:
        raw=rec.get('raw_data') or {}
        nd=rec.get('normalized_data') or {}
        if not any(_meaningful(v) for v in raw.values()) and not any(comparable(v) for v in nd.values()):
            flags.append('EXTRACTION_JSON_VIDE')
        if relevant_critical_missing(rec,business_context):
            flags.append('CRITICAL_FIELD_MISSING')
    # PARTIELLE / LOW_FILL_RATE de la Partie 1 restent traçables dans DOCUMENTS,
    # mais ne déclenchent plus à eux seuls une validation manuelle.
    return list(dict.fromkeys(flags))


def field_evidence(processed_records, field):
    ev=[]
    for r in processed_records:
        nd=r.get('normalized_data') or {}
        tr=r.get('normalization_trace') or {}
        if field in nd:
            ev.append({
                'doc_type':r.get('doc_type'),
                'page':r.get('page_num'),
                'value':nd.get(field),
                'raw':(r.get('raw_data') or {}).get(field),
                'trace':tr.get(field) or {},
                'classification_review_required':bool(r.get('classification_review_required')),
                'quality_flags':penalizing_quality_flags(r),
                'extraction_status':r.get('extraction_status'),
            })
    return ev


def anomaly_applies_to_field(anomaly, field, page=None, doc_type=None):
    """
    Rend cohérents CHAMPS_DETAIL, ANOMALIES et le statut du dossier.
    """
    typ=anomaly.get('TYPE_ANOMALIE')
    achamp=anomaly.get('CHAMP')
    apage=anomaly.get('PAGE')

    if typ=='FORMAT_REVIEW':
        return achamp==field and (apage is None or page==apage)

    if typ in {'CRITICAL_FIELD_MISSING','CRITICAL_FIELD_MISSING_POSTPROCESS'}:
        parts={x.strip() for x in str(achamp or '').split('|') if x.strip()}
        return field in parts

    if typ=='CROSS_DOCUMENT_CONFLICT':
        return field in CROSSCHECK_FIELD_MAP.get(str(achamp),set())

    if typ=='INVALID_DATE_ORDER':
        return field in DATE_ORDER_FIELDS.get(str(achamp),set())

    if typ in {
        'CLASSIFICATION_REVIEW_REQUIRED',
        'EXTRACTION_JSON_VIDE',
        'EXTRACTION_PARTIELLE',
    }:
        return apage is None or page==apage

    return False


def calculate_field_confidence(processed_records, business_context, anomalies):
    rows=[]

    for field in ALL_FIELDS:
        applicability,importance=field_policy(field,business_context)
        ev=field_evidence(processed_records,field)
        present=[e for e in ev if comparable(e['value'])]

        if importance=='INFO_ONLY':
            rows.append({
                'CHAMP':field,
                'APPLICABILITE':applicability,
                'DQ_IMPORTANCE':importance,
                'FIELD_CONFIDENCE':None,
                'FIELD_CONFIDENCE_LEVEL':'INFO_ONLY',
                'FIELD_CONFIDENCE_REASON':'HORS_SCORE_DQ',
                'NB_OCCURRENCES':len(present),
            })
            continue

        if not present:
            score=0
            reasons=['AUCUNE_VALEUR_NORMALISEE']
        else:
            score=70
            reasons=['VALEUR_PRESENTE']

            statuses=[(e['trace'] or {}).get('status') for e in present]
            if any(s=='REVIEW' for s in statuses):
                score-=35
                reasons.append('NORMALISATION_REVIEW')
            elif all(s in {'RAW_OK','AUTO_OK'} for s in statuses):
                score+=10
                reasons.append('NORMALISATION_DETERMINISTE')

            if any(e['classification_review_required'] for e in present):
                score-=30
                reasons.append('CLASSIFICATION_REVIEW')

            if any(str(e['extraction_status'] or '').upper() in
                   {'JSON_VIDE','FAILED','ECHEC'} for e in present):
                score-=20
                reasons.append('EXTRACTION_ECHEC_OU_VIDE')

            if any(e['quality_flags'] for e in present):
                score-=5
                reasons.append('QUALITY_FLAG_PRESENT')

            # IMPORTANT V3 :
            # conflit entre champs équivalents de documents différents.
            relevant_conflicts=[
                a for a in anomalies
                if a.get('TYPE_ANOMALIE')=='CROSS_DOCUMENT_CONFLICT'
                and field in CROSSCHECK_FIELD_MAP.get(str(a.get('CHAMP')),set())
            ]
            if relevant_conflicts:
                score-=40
                reasons.append(
                    'CROSS_DOCUMENT_CONFLICT:' +
                    '|'.join(sorted({str(a.get('CHAMP')) for a in relevant_conflicts}))
                )

            if any(
                a.get('TYPE_ANOMALIE')=='INVALID_DATE_ORDER'
                and anomaly_applies_to_field(a,field)
                for a in anomalies
            ):
                score-=40
                reasons.append('INVALID_DATE_ORDER')

            vals=[e['value'] for e in present]
            if len(vals)>=2:
                kind=FIELD_TYPES.get(field,'text')
                if all(same_value(kind,vals[0],x) for x in vals[1:]):
                    score+=20
                    reasons.append(f'CONCORDANCE_{len(vals)}_LECTURES')
                else:
                    score-=35
                    reasons.append('DIVERGENCE_INTER_OCCURRENCES')

            score=max(0,min(100,int(round(score))))

        level='HIGH' if score>=90 else ('MEDIUM' if score>=70 else ('LOW' if score>0 else 'MISSING'))

        rows.append({
            'CHAMP':field,
            'APPLICABILITE':applicability,
            'DQ_IMPORTANCE':importance,
            'FIELD_CONFIDENCE':score,
            'FIELD_CONFIDENCE_LEVEL':level,
            'FIELD_CONFIDENCE_REASON':' | '.join(reasons),
            'NB_OCCURRENCES':len(present),
        })

    return rows


def add_postprocess_critical_missing(processed_records, anomalies, business_context, source):
    """Ajoute MISSING seulement si le champ est réellement absent.

    Si le RAW est présent mais ambigu, FORMAT_REVIEW porte l'anomalie afin d'éviter
    le doublon FORMAT_REVIEW + CRITICAL_FIELD_MISSING.
    """
    existing={(a.get('TYPE_ANOMALIE'),str(a.get('CHAMP'))) for a in anomalies}
    for field in CRITICAL_FIELDS:
        if not is_dq_relevant_field(field,business_context):
            continue
        owner=None; owner_records=[]
        for dt,fields in FIELD_SCHEMA.items():
            if field in fields:
                owner=dt
                owner_records=[r for r in processed_records if r.get('doc_type')==dt]
                break
        if not owner_records:
            continue
        if any(comparable((r.get('normalized_data') or {}).get(field)) for r in owner_records):
            continue
        if any(_meaningful((r.get('raw_data') or {}).get(field)) for r in owner_records):
            continue
        key=('CRITICAL_FIELD_MISSING_POSTPROCESS',field)
        if key in existing:
            continue
        anomalies.append({
            'FICHIER':source,'PAGE':owner_records[0].get('page_num'),'TYPE_DOCUMENT':owner,
            'TYPE_ANOMALIE':'CRITICAL_FIELD_MISSING_POSTPROCESS','SEVERITE':'BLOQUANT',
            'CHAMP':field,'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
            'MOTIF':'champ critique réellement absent après normalisation Partie 2'
        })


def add_business_controls(processed, anomalies, business_context):
    records=processed['page_records']

    values={}
    for r in records:
        for f,v in (r.get('normalized_data') or {}).items():
            if comparable(v) and f not in values:
                values[f]=v

    date_pairs=[
        ('DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','CONTRAT'),
        ('CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTR'),
        ('CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTS'),
        ('TTR_DATE_DEBUT','TTR_DATE_FIN','PERMIS_TTR'),
    ]
    for f1,f2,label in date_pairs:
        d1=parse_date_safe(values.get(f1))
        d2=parse_date_safe(values.get(f2))
        if d1 and d2 and d2<d1:
            anomalies.append({
                'FICHIER':processed['source_file'],
                'PAGE':None,
                'TYPE_DOCUMENT':'MULTI',
                'TYPE_ANOMALIE':'INVALID_DATE_ORDER',
                'SEVERITE':'BLOQUANT',
                'CHAMP':label,
                'VALEUR_RAW':None,
                'VALEUR_NORMALISEE':f'{values.get(f1)} > {values.get(f2)}',
                'MOTIF':'date fin antérieure à date début'
            })

    # V5 : aucune anomalie sur la date d'effet d'augmentation ici.
    # La période d'effet sera fournie par le fichier annuel de paramétrage du planning TL.


def dedupe_anomalies(anomalies):
    seen=set()
    out=[]
    for a in anomalies:
        key=(
            a.get('FICHIER'),a.get('PAGE'),a.get('TYPE_DOCUMENT'),
            a.get('TYPE_ANOMALIE'),a.get('SEVERITE'),a.get('CHAMP'),
            str(a.get('VALEUR_NORMALISEE')),a.get('MOTIF')
        )
        if key not in seen:
            seen.add(key)
            out.append(a)
    return out


def calculate_dq_summary(processed, field_conf_rows):
    records=processed['page_records']
    anomalies=processed.get('anomalies') or []
    business_context=processed.get('business_context') or {}

    field_map={x['CHAMP']:x for x in field_conf_rows}

    critical_scores=[]
    for f in CRITICAL_FIELDS:
        if not is_dq_relevant_field(f,business_context):
            continue
        owner=None
        for dt,fields in FIELD_SCHEMA.items():
            if f in fields:
                owner=dt
                break
        if owner and any(r.get('doc_type')==owner for r in records):
            score=field_map[f].get('FIELD_CONFIDENCE')
            critical_scores.append(1 if score not in (None,0) else 0)

    critical_completeness=(
        sum(critical_scores)/len(critical_scores)
        if critical_scores else 0
    )

    cross_conflicts=[
        a for a in anomalies
        if a.get('TYPE_ANOMALIE')=='CROSS_DOCUMENT_CONFLICT'
    ]
    cross_checks_possible=0
    for g in CROSS_DOCUMENT_GROUPS:
        if not crosscheck_group_applicable(g,business_context):
            continue
        vals=[]
        for dt,f in g['fields'].items():
            for r in records:
                if r.get('doc_type')==dt:
                    v=(r.get('normalized_data') or {}).get(f)
                    if comparable(v):
                        vals.append(v)
        if len(vals)>=2:
            cross_checks_possible+=1

    cross_document=(
        1.0 if cross_checks_possible==0
        else max(0,1-(len(cross_conflicts)/cross_checks_possible))
    )

    traces=[]
    for r in records:
        for field,tr in (r.get('normalization_trace') or {}).items():
            if is_dq_relevant_field(field,business_context):
                traces.append(tr)
    nonmissing=[t for t in traces if t.get('status')!='MISSING']
    normalization=(
        sum(t.get('status') in {'RAW_OK','AUTO_OK'} for t in nonmissing)/len(nonmissing)
        if nonmissing else 0
    )

    business_bad=sum(
        a.get('TYPE_ANOMALIE') in {'INVALID_DATE_ORDER'}
        for a in anomalies
    )
    business_validity=1.0 if business_bad==0 else 0.0

    page_flags=[page_regulatory_flags(r,business_context) for r in records]
    severe_pages=sum(
        any(x in {'EXTRACTION_JSON_VIDE','EXTRACTION_PARTIELLE','CRITICAL_FIELD_MISSING'} for x in fl)
        for fl in page_flags
    )
    extraction_quality=max(0,1-(severe_pages/max(1,len(records))))

    class_review=sum(
        'CLASSIFICATION_REVIEW_REQUIRED' in fl
        for fl in page_flags
    )
    classification_quality=max(0,1-(class_review/max(1,len(records))))

    components={
        'critical_completeness':critical_completeness,
        'cross_document':cross_document,
        'normalization':normalization,
        'business_validity':business_validity,
        'extraction_quality':extraction_quality,
        'classification_quality':classification_quality,
    }

    dq_score=round(
        sum(components[k]*DQ_WEIGHTS[k] for k in DQ_WEIGHTS),1
    )

    blocking=[a for a in anomalies if a.get('SEVERITE')=='BLOQUANT']
    review=[a for a in anomalies if a.get('SEVERITE')=='REVIEW']

    # V3 : décision cohérente avec l'onglet ANOMALIES.
    # Une anomalie REVIEW empêche AUTO_OK.
    if blocking:
        validation='BLOCKED'
    elif review:
        validation='REVIEW'
    elif not critical_scores or critical_completeness < 1.0:
        validation='REVIEW'
    else:
        validation='AUTO_OK'

    hard_reasons=list(dict.fromkeys(
        str(a.get('TYPE_ANOMALIE'))
        for a in blocking
    ))

    level='HIGH' if dq_score>=90 else ('MEDIUM' if dq_score>=70 else 'LOW')

    return {
        'TYPE_DOSSIER':business_context.get('TYPE_DOSSIER'),
        'CTS_DATE_DOCUMENT_AUGMENTATION':business_context.get('CTS_DATE_DOCUMENT_AUGMENTATION'),
        'DQ_SCORE':dq_score,
        'DQ_LEVEL':level,
        'VALIDATION_AUTO':validation,
        'HARD_BLOCK':bool(blocking),
        'HARD_BLOCK_REASONS':' | '.join(hard_reasons),
        'NB_BLOQUANTS':len(blocking),
        'NB_REVIEW':len(review),
        'SCORE_CRITICAL_COMPLETENESS':round(critical_completeness*100,1),
        'SCORE_CROSS_DOCUMENT':round(cross_document*100,1),
        'SCORE_NORMALIZATION':round(normalization*100,1),
        'SCORE_BUSINESS_VALIDITY':round(business_validity*100,1),
        'SCORE_EXTRACTION_QUALITY':round(extraction_quality*100,1),
        'SCORE_CLASSIFICATION_QUALITY':round(classification_quality*100,1),
    }


def enrich_field_rows(field_rows, anomalies, field_confidence, dossier_status):
    """
    Ajoute au CHAMPS_DETAIL un statut FINAL cohérent avec les autres feuilles.
    """
    conf_map={x['CHAMP']:x for x in field_confidence}

    for row in field_rows:
        field=row['CHAMP']
        page=row.get('PAGE')
        dt=row.get('TYPE_DOCUMENT')
        importance=row.get('DQ_IMPORTANCE')

        conf=conf_map.get(field,{})
        row['FIELD_CONFIDENCE']=conf.get('FIELD_CONFIDENCE')
        row['FIELD_CONFIDENCE_LEVEL']=conf.get('FIELD_CONFIDENCE_LEVEL')
        row['FIELD_CONFIDENCE_REASON']=conf.get('FIELD_CONFIDENCE_REASON')

        if importance=='INFO_ONLY':
            row['DQ_STATUS_FIELD']='INFO_ONLY'
            row['DQ_SEVERITY']='INFO'
            row['DQ_ANOMALIES']=''
            row['DQ_REASON']='HORS_SCORE_DQ'
            row['DOSSIER_STATUS']=dossier_status
            continue

        related=[
            a for a in anomalies
            if anomaly_applies_to_field(a,field,page,dt)
        ]

        if related:
            maxsev=max(
                (a.get('SEVERITE','REVIEW') for a in related),
                key=lambda x:SEVERITY_RANK.get(x,1)
            )
            row['DQ_STATUS_FIELD']='BLOCKED' if maxsev=='BLOQUANT' else 'REVIEW'
            row['DQ_SEVERITY']=maxsev
            row['DQ_ANOMALIES']=' | '.join(dict.fromkeys(
                str(a.get('TYPE_ANOMALIE')) for a in related
            ))
            row['DQ_REASON']=' | '.join(dict.fromkeys(
                str(a.get('MOTIF')) for a in related if a.get('MOTIF')
            ))
        elif row.get('NORMALIZATION_STATUS')=='MISSING':
            row['DQ_STATUS_FIELD']='MISSING'
            row['DQ_SEVERITY']='INFO'
            row['DQ_ANOMALIES']=''
            row['DQ_REASON']='VALEUR_ABSENTE_NON_BLOQUANTE'
        else:
            row['DQ_STATUS_FIELD']='OK'
            row['DQ_SEVERITY']='OK'
            row['DQ_ANOMALIES']=''
            row['DQ_REASON']='AUCUNE_ANOMALIE_DQ'

        row['DOSSIER_STATUS']=dossier_status

    return field_rows


def build_crosscheck_rows(processed):
    rows=[]
    by_type=defaultdict(list)
    for r in processed['page_records']:
        by_type[r.get('doc_type')].append(r)

    for group in CROSS_DOCUMENT_GROUPS:
        if not crosscheck_group_applicable(group, processed.get('business_context') or {}):
            rows.append({
                'FICHIER':processed['source_file'],
                'TYPE_DOSSIER':(processed.get('business_context') or {}).get('TYPE_DOSSIER'),
                'CONTROLE':group['name'],
                'STATUS_CONTROLE':'NON_APPLICABLE_AUGMENTATION',
                'SEVERITE':'INFO','NB_VALEURS':0,
                'DETAIL':'écart attendu sur une augmentation; contrôle non bloquant'
            })
            continue
        vals=[]
        for dt,field in group['fields'].items():
            for r in by_type.get(dt,[]):
                v=(r.get('normalized_data') or {}).get(field)
                if comparable(v):
                    vals.append({
                        'doc_type':dt,
                        'field':field,
                        'page':r.get('page_num'),
                        'value':v,
                        'raw':(r.get('raw_data') or {}).get(field),
                    })

        if len(vals)<2:
            status='NON_CONTROLE'
            severity='INFO'
        else:
            base=vals[0]['value']
            conflict=any(
                not same_value(group['kind'],base,x['value'])
                for x in vals[1:]
            )
            if conflict:
                status='CONFLIT'
                severity='BLOQUANT' if group['name'] in {
                    'SALAIRE_NET','NUMERO_PERMIS',
                    'DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'
                } else 'REVIEW'
            else:
                status='OK'
                severity='OK'

        rows.append({
            'FICHIER':processed['source_file'],
            'TYPE_DOSSIER':(processed.get('business_context') or {}).get('TYPE_DOSSIER'),
            'CONTROLE':group['name'],
            'STATUS_CONTROLE':status,
            'SEVERITE':severity,
            'NB_VALEURS':len(vals),
            'DETAIL':' | '.join(
                f"{x['doc_type']}.{x['field']}@p{x['page']}={x['value']}"
                for x in vals
            )
        })
    return rows



# =====================================================================
# V4 — PROPOSITION DE VALEUR + CONTROLE MEME CLIENT + PERMIS RTL
# =====================================================================

DOCUMENT_PRIORITY = {
    'CONTRAT_SPECIFIQUE': 1,
    'CONTRAT_TRAVAIL': 2,
    'TITRE_TRAVAIL': 3,
    'ENGAGEMENT_DOMICILIATION': 4,
    'PERMIS_TRAVAIL_COUVERTURE': 99,
}

EQUIVALENT_FIELD_GROUPS = {}
for _g in CROSS_DOCUMENT_GROUPS:
    for _f in _g['fields'].values():
        EQUIVALENT_FIELD_GROUPS[_f] = {
            'name': _g['name'],
            'kind': _g['kind'],
            'fields': dict(_g['fields']),
        }


def _identity_text(v):
    if v in (None, ''):
        return None
    s = str(v).replace('\u00a0', ' ').strip().upper()
    return re.sub(r'[^A-Z0-9]', '', s) or None


def _identity_reference(v):
    if v in (None, ''):
        return None
    s = str(v).replace('\u00a0', ' ').strip().upper()
    s = re.sub(r'\s*([/\-])\s*', r'\1', re.sub(r'\s+', ' ', s)).strip()
    return re.sub(r'[^A-Z0-9]', '', s) or None


def identity_signature(rec):
    dt = rec.get('doc_type')
    nd = rec.get('normalized_data') or {}

    name = dob = permit = None

    if dt == 'CONTRAT_SPECIFIQUE':
        name = nd.get('CTS_NOM_PRENOM_TRAVAILLEUR')
        dob = nd.get('CTS_DATE_NAISSANCE')
        permit = nd.get('CTS_NUMERO_PERMIS_TRAVAIL')

    elif dt == 'CONTRAT_TRAVAIL':
        name = nd.get('CTR_NOM_PRENOM_TRAVAILLEUR')
        dob = nd.get('CTR_DATE_NAISSANCE')
        permit = nd.get('CTR_NUMERO_PERMIS_TRAVAIL')

    elif dt == 'TITRE_TRAVAIL':
        name = ' '.join(
            x for x in [
                str(nd.get('TTR_NOM') or '').strip(),
                str(nd.get('TTR_PRENOM') or '').strip(),
            ] if x
        ) or None
        dob = nd.get('TTR_DATE_NAISSANCE')
        permit = nd.get('TTR_NUMERO_PERMIS')

    elif dt == 'ENGAGEMENT_DOMICILIATION':
        name = nd.get('DOM_NOM_RAISON_SOCIAL_CLIENT')

    return {
        'name': _identity_text(name),
        'dob': str(dob) if comparable(dob) else None,
        'permit': _identity_reference(permit) if comparable(permit) else None,
    }


def compare_identity_signatures(a, b):
    matches, conflicts = [], []

    for key in ('permit', 'dob', 'name'):
        av = (a or {}).get(key)
        bv = (b or {}).get(key)

        if av in (None, '') or bv in (None, ''):
            continue

        if key == 'name':
            ok = (av == bv) or (av in bv) or (bv in av)
        else:
            ok = (av == bv)

        (matches if ok else conflicts).append(key)

    return matches, conflicts


def confirm_same_client(candidate_rec, all_records):
    """
    Confirmation par au moins un AUTRE document :
      - même permis, ou
      - même nom + même date de naissance.
    Une contradiction sur permis/date de naissance empêche la proposition.
    Pour DOM, un nom concordant est accepté comme preuve plus faible.
    """
    cand_sig = identity_signature(candidate_rec)
    confirmations = []
    contradictions = []

    for other in all_records:
        if other is candidate_rec:
            continue
        if other.get('doc_type') == 'PERMIS_TRAVAIL_COUVERTURE':
            continue

        matches, conflicts = compare_identity_signatures(
            cand_sig, identity_signature(other)
        )

        if any(x in {'permit', 'dob'} for x in conflicts):
            contradictions.append(
                f"{other.get('doc_type')}@p{other.get('page_num')}:" +
                ",".join(conflicts)
            )
            continue

        strong = (
            'permit' in matches
            or ('name' in matches and 'dob' in matches)
        )

        if (
            candidate_rec.get('doc_type') == 'ENGAGEMENT_DOMICILIATION'
            and 'name' in matches
            and not conflicts
        ):
            strong = True

        if strong:
            confirmations.append(
                f"{other.get('doc_type')}@p{other.get('page_num')}:" +
                ",".join(matches)
            )

    if contradictions:
        return {
            'status': 'CLIENT_MISMATCH',
            'evidence': ' | '.join(contradictions),
        }

    if confirmations:
        return {
            'status': 'SAME_CLIENT_CONFIRMED',
            'evidence': ' | '.join(confirmations),
        }

    return {
        'status': 'SAME_CLIENT_UNCONFIRMED',
        'evidence': 'Pas assez de preuves croisées',
    }


def candidate_fields_for_target(target_field):
    group = EQUIVALENT_FIELD_GROUPS.get(target_field)
    if not group:
        return []

    items = list(group['fields'].items())
    items.sort(key=lambda x: DOCUMENT_PRIORITY.get(x[0], 99))
    return items


def propose_best_value(processed, target_field, target_page=None, target_doc_type=None):
    """
    Priorité : CTS > CTR > TTR > DOM.
    La cellule en anomalie ne se propose pas elle-même.
    """
    records = processed.get('page_records') or []
    candidates = []

    for dt, field in candidate_fields_for_target(target_field):
        for rec in records:
            if rec.get('doc_type') != dt:
                continue

            value = (rec.get('normalized_data') or {}).get(field)
            tr = (rec.get('normalization_trace') or {}).get(field) or {}

            if not comparable(value) or tr.get('status') == 'REVIEW':
                continue

            same_cell = (
                target_page is not None
                and rec.get('page_num') == target_page
                and target_doc_type == dt
                and field == target_field
            )
            if same_cell:
                continue

            client = confirm_same_client(rec, records)
            if client['status'] != 'SAME_CLIENT_CONFIRMED':
                continue

            candidates.append({
                'priority': DOCUMENT_PRIORITY.get(dt, 99),
                'value': value,
                'source_document': dt,
                'source_page': rec.get('page_num'),
                'source_field': field,
                'same_client_status': client['status'],
                'same_client_evidence': client['evidence'],
            })

    if not candidates:
        return {
            'value': None,
            'source_document': None,
            'source_page': None,
            'source_field': None,
            'same_client_status': 'NO_SAFE_PROPOSAL',
            'same_client_evidence': 'Aucune source confirmée comme même client',
            'rule': 'NO_SAFE_PROPOSAL',
        }

    candidates.sort(key=lambda x: x['priority'])
    best = candidates[0]

    return {
        'value': best['value'],
        'source_document': best['source_document'],
        'source_page': best['source_page'],
        'source_field': best['source_field'],
        'same_client_status': best['same_client_status'],
        'same_client_evidence': best['same_client_evidence'],
        'rule': 'PRIORITY_CTS_CTR_TTR_DOM_WITH_SAME_CLIENT_GATE',
    }


def reverse_reference_group_order(value):
    """
    Inverse uniquement l'ordre des GROUPES, jamais les chiffres d'un groupe.

    000395-26-31/00028138-20
      -> 20-00028138/31-26-000395
    """
    if not comparable(value):
        return None

    s = str(value).strip()
    groups = re.split(r'[/\-]+', s)
    seps = re.findall(r'[/\-]+', s)

    if len(groups) < 2 or len(seps) != len(groups) - 1:
        return None

    groups = list(reversed(groups))
    seps = list(reversed(seps))

    out = groups[0]
    for sep, group in zip(seps, groups[1:]):
        out += sep + group

    return out


def resolve_ttr_permit_rtl(processed_records):
    """
    Corrige TTR_NUMERO_PERMIS seulement si :
      - CTS/CTR donnent le même numéro de permis ;
      - le TTR inversé par GROUPES correspond exactement ;
      - TTR et CTS/CTR ont le même nom + date de naissance.

    Le permis TTR courant est précisément le champ suspect :
    son conflit n'est donc PAS utilisé pour rejeter le test d'identité.
    """
    trusted = []

    for rec in processed_records:
        dt = rec.get('doc_type')
        nd = rec.get('normalized_data') or {}

        if dt == 'CONTRAT_SPECIFIQUE':
            v = nd.get('CTS_NUMERO_PERMIS_TRAVAIL')
        elif dt == 'CONTRAT_TRAVAIL':
            v = nd.get('CTR_NUMERO_PERMIS_TRAVAIL')
        else:
            continue

        if comparable(v):
            trusted.append((dt, rec, v, _identity_reference(v)))

    if not trusted:
        return []

    trusted_keys = {x[3] for x in trusted if x[3]}
    if len(trusted_keys) != 1:
        # CTR/CTS eux-mêmes sont incohérents : ne rien corriger.
        return []

    canonical_key = next(iter(trusted_keys))
    trusted.sort(key=lambda x: DOCUMENT_PRIORITY.get(x[0], 99))
    canonical_value = trusted[0][2]

    corrections = []

    for rec in processed_records:
        if rec.get('doc_type') != 'TITRE_TRAVAIL':
            continue

        nd = rec.get('normalized_data') or {}
        raw = rec.get('raw_data') or {}
        trace = rec.get('normalization_trace') or {}

        current = nd.get('TTR_NUMERO_PERMIS')
        if not comparable(current):
            continue

        if _identity_reference(current) == canonical_key:
            continue

        rtl_candidate = reverse_reference_group_order(current)
        if not rtl_candidate:
            continue

        if _identity_reference(rtl_candidate) != canonical_key:
            continue

        ttr_sig = identity_signature(rec)
        client_ok = False
        evidence = []

        for _, trusted_rec, _, _ in trusted:
            matches, conflicts = compare_identity_signatures(
                ttr_sig, identity_signature(trusted_rec)
            )

            # On IGNORE le conflit sur "permit" car c'est précisément
            # le champ dont on teste l'orientation RTL.
            if 'dob' in conflicts or 'name' in conflicts:
                continue

            if 'name' in matches and 'dob' in matches:
                client_ok = True
                evidence.append(
                    f"{trusted_rec.get('doc_type')}@p{trusted_rec.get('page_num')}:name,dob"
                )

        if not client_ok:
            continue

        previous = current
        nd['TTR_NUMERO_PERMIS'] = canonical_value

        trace['TTR_NUMERO_PERMIS'] = {
            'raw': raw.get('TTR_NUMERO_PERMIS'),
            'normalized': canonical_value,
            'status': 'AUTO_OK',
            'rule': 'TTR_PERMIT_RTL_GROUP_ORDER_RESOLVED_BY_CROSSCHECK',
            'changed': canonical_value != raw.get('TTR_NUMERO_PERMIS'),
            'message': (
                'Ordre des groupes du permis TTR inversé droite->gauche; '
                'résolu par CTR/CTS + concordance nom/date naissance'
            ),
            'previous_normalized': previous,
            'rtl_candidate': rtl_candidate,
            'identity_evidence': ' | '.join(evidence),
        }

        corrections.append({
            'page': rec.get('page_num'),
            'raw': raw.get('TTR_NUMERO_PERMIS'),
            'previous': previous,
            'normalized': canonical_value,
            'rule': 'TTR_PERMIT_RTL_GROUP_ORDER_RESOLVED_BY_CROSSCHECK',
        })

    return corrections



def build_manual_validation_rows(processed, field_rows):
    dq = processed.get('dq_summary') or {}
    ctx = processed.get('business_context') or {}
    out = []

    for r in field_rows:
        if r.get('DQ_STATUS_FIELD') not in {'REVIEW', 'BLOCKED'}:
            continue

        proposal = propose_best_value(
            processed,
            r.get('CHAMP'),
            r.get('PAGE'),
            r.get('TYPE_DOCUMENT'),
        )

        out.append({
            'FICHIER': r.get('FICHIER'),
            'TYPE_DOSSIER': ctx.get('TYPE_DOSSIER'),
            'DOSSIER_STATUS': dq.get('VALIDATION_AUTO'),
            'DQ_SCORE': dq.get('DQ_SCORE'),
            'SEVERITE': r.get('DQ_SEVERITY'),
            'PAGE': r.get('PAGE'),
            'TYPE_DOCUMENT': r.get('TYPE_DOCUMENT'),
            'CHAMP': r.get('CHAMP'),
            'RAW': r.get('RAW'),
            'NORMALIZED': r.get('NORMALIZED'),
            'NORMALIZATION_STATUS': r.get('NORMALIZATION_STATUS'),
            'FIELD_CONFIDENCE': r.get('FIELD_CONFIDENCE'),
            'TYPE_ANOMALIE': r.get('DQ_ANOMALIES'),
            'MOTIF': r.get('DQ_REASON'),

            'VALEUR_PROPOSEE': proposal.get('value'),
            'SOURCE_PROPOSITION_DOCUMENT': proposal.get('source_document'),
            'SOURCE_PROPOSITION_PAGE': proposal.get('source_page'),
            'SOURCE_PROPOSITION_CHAMP': proposal.get('source_field'),
            'MEME_CLIENT_STATUT': proposal.get('same_client_status'),
            'MEME_CLIENT_PREUVE': proposal.get('same_client_evidence'),
            'REGLE_PROPOSITION': proposal.get('rule'),

            'VALEUR_VALIDEE': None,
            'DECISION_MANUELLE': None,
            'COMMENTAIRE': None,
        })

    return out



print('✅ DQ V5 chargé : proposition priorisée + même client + permis RTL + CarthagoDom')


## 6A. V5 — Rapprochement CarthagoDom et validation du numéro de domiciliation

Le rapprochement est volontairement conservateur :

1. `SIEGE_RACINE` est extrait de `DOM_COMPTE_LOCAL` par regex `07310` + 6 chiffres ;
2. le match principal est l'égalité exacte avec `Client` Carthago ;
3. la similarité du nom est **informationnelle uniquement** ;
4. une ligne PREDOM seule n'est jamais proposée comme domiciliation finale ;
5. en cas de plusieurs DOM possibles sans correspondance exacte de référence, aucune valeur n'est imposée.


### Note V5.5 — compte local et Modulo 97

Le contrôle `SIEGE_RACINE` est une règle métier Tosyali/Carthago et reste basé sur la regex `07310 + 6 chiffres`. Le contrôle Modulo 97 est exécuté sur les codes intérieurs BNP/Algérie : code agence (5 chiffres, dont les zéros initiaux sont conservés) + numéro de compte (10 chiffres) + clé (2 chiffres). Le code banque 027 est exclu du calcul de la clé. Aucune valeur du compte ni de la clé n’est modifiée automatiquement.


**Correction V5.5.** La V5.3 calculait à tort `int(RIB_20_CHIFFRES) % 97`, ce qui incluait le code banque `027` et produisait des faux `CLE_MODULO97_INVALIDE`. La V5.5 contrôle la clé sur `CODE_AGENCE(5) + NUMERO_COMPTE(10) + CLE(2)` et calcule la clé attendue sur `CODE_AGENCE(5) + NUMERO_COMPTE(10)`.


In [ ]:

import unicodedata
from difflib import SequenceMatcher

try:
    from rapidfuzz import fuzz as _rapidfuzz_fuzz
except Exception:
    _rapidfuzz_fuzz = None


def _text_key(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    s = unicodedata.normalize('NFKD', str(value))
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper().strip()
    s = re.sub(r'[^A-Z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_name_for_match(value):
    return _text_key(value)


def name_similarity(a, b):
    """Score 0..100. Informatif uniquement; jamais utilisé comme clé primaire."""
    a = normalize_name_for_match(a)
    b = normalize_name_for_match(b)
    if not a or not b:
        return None
    if _rapidfuzz_fuzz is not None:
        return round(float(_rapidfuzz_fuzz.token_sort_ratio(a, b)), 1)
    aa = ' '.join(sorted(a.split()))
    bb = ' '.join(sorted(b.split()))
    return round(100.0 * SequenceMatcher(None, aa, bb).ratio(), 1)


def extract_siege_racine(numero_compte):
    """
    Extrait exactement : 07310 + 6 chiffres.
    Exemple 02700731010910800156 -> 07310109108.
    """
    if numero_compte is None:
        return None
    s = str(numero_compte).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.', 1)[0]
    digits = re.sub(r'\D', '', s)
    matches = re.findall(r'07310\d{6}', digits)
    uniq = list(dict.fromkeys(matches))
    return uniq[0] if len(uniq) == 1 else None


def normalize_dom_compte_local(value):
    """Retourne uniquement les chiffres du compte local, sans inventer/corriger."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.', 1)[0]
    digits = re.sub(r'\D', '', s)
    return digits or None


def analyze_dom_compte_local(value):
    """
    Contrôles séparés :
      - règles métier Tosyali : préfixe 02700, regex 07310 + 6 chiffres,
        ordinal 001 juste après le motif ;
      - contrôle bancaire Modulo 97 si le compte est un RIB numérique de 20 chiffres.

    IMPORTANT : pour les RIB BNP/Algérie de 20 chiffres, la clé est contrôlée sur les
    codes intérieurs : code agence 5 chiffres + numéro de compte 10 chiffres + clé 2 chiffres.
    Le code banque (3 chiffres, ex. 027) n'entre pas dans le calcul de cette clé.
    Le découpage métier Tosyali SIEGE_RACINE reste indépendant.
    """
    digits = normalize_dom_compte_local(value)
    out = {
        'DOM_COMPTE_LOCAL_NORMALIZED': digits,
        'COMPTE_LONGUEUR': len(digits) if digits else None,
        'COMPTE_20_CHIFFRES': bool(digits and len(digits) == 20),
        'PREFIXE_02700_OK': bool(digits and digits.startswith('02700')),
        'SIEGE_RACINE': None,
        'RACINE_6': None,
        'ORDINAL_APRES_RACINE': None,
        'ORDINAL_001_OK': False,
        'CODE_BANQUE_RIB': digits[:3] if digits and len(digits) >= 3 else None,
        'CODE_AGENCE_RIB': digits[3:8] if digits and len(digits) >= 8 else None,
        'NUMERO_COMPTE_RIB': digits[8:18] if digits and len(digits) >= 18 else None,
        'CLE_COMPTE': digits[-2:] if digits and len(digits) >= 2 else None,
        'CLE_ATTENDUE_MOD97': None,
        'MODULO97_RESTE': None,
        'MODULO97_OK': None,
        'STRUCTURE_TOSYALI_OK': False,
        'DOM_COMPTE_LOCAL_STATUS': None,
        'DOM_COMPTE_LOCAL_REQUIRES_MANUAL': True,
    }
    if not digits:
        out['DOM_COMPTE_LOCAL_STATUS'] = 'COMPTE_ABSENT'
        return out

    matches = list(re.finditer(r'07310(?P<racine>\d{6})', digits))
    if len(matches) == 1:
        m = matches[0]
        out['SIEGE_RACINE'] = m.group(0)
        out['RACINE_6'] = m.group('racine')
        ordinal = digits[m.end():m.end()+3]
        out['ORDINAL_APRES_RACINE'] = ordinal or None
        out['ORDINAL_001_OK'] = ordinal == '001'

    out['STRUCTURE_TOSYALI_OK'] = bool(
        out['COMPTE_20_CHIFFRES']
        and out['PREFIXE_02700_OK']
        and out['SIEGE_RACINE']
        and out['ORDINAL_001_OK']
    )

    # Modulo 97 — clé des codes intérieurs.
    # RIB 20 chiffres : banque(3) + agence(5) + compte(10) + clé(2).
    # Pour la clé de compte BNP/Algérie, on contrôle : agence + compte + clé,
    # sans inclure le code banque.
    if out['COMPTE_20_CHIFFRES']:
        code_banque = digits[0:3]
        code_agence = digits[3:8]
        numero_compte = digits[8:18]
        cle_lue = digits[18:20]

        interior_base = code_agence + numero_compte
        interior_full = interior_base + cle_lue

        out['CODE_BANQUE_RIB'] = code_banque
        out['CODE_AGENCE_RIB'] = code_agence
        out['NUMERO_COMPTE_RIB'] = numero_compte
        out['CLE_COMPTE'] = cle_lue
        out['MODULO97_RESTE'] = int(interior_full) % 97
        out['MODULO97_OK'] = out['MODULO97_RESTE'] == 0

        expected = 97 - ((int(interior_base) * 100) % 97)
        out['CLE_ATTENDUE_MOD97'] = f'{expected:02d}'

    if not out['STRUCTURE_TOSYALI_OK']:
        out['DOM_COMPTE_LOCAL_STATUS'] = 'STRUCTURE_TOSYALI_A_REVOIR'
    elif out['MODULO97_OK'] is False:
        out['DOM_COMPTE_LOCAL_STATUS'] = 'CLE_MODULO97_INVALIDE'
    elif out['MODULO97_OK'] is True:
        out['DOM_COMPTE_LOCAL_STATUS'] = 'STRUCTURE_ET_MODULO97_OK'
        out['DOM_COMPTE_LOCAL_REQUIRES_MANUAL'] = False
    else:
        out['DOM_COMPTE_LOCAL_STATUS'] = 'MODULO97_NON_CALCULABLE'

    return out


def normalize_carthago_client(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.', 1)[0]
    digits = re.sub(r'\D', '', s)
    # Excel peut supprimer le zéro initial si Client a été stocké comme nombre.
    if len(digits) == 10 and digits.startswith('7310'):
        digits = '0' + digits
    return digits or None


def _header_key(value):
    return _text_key(value)


def detect_column(columns, exact=None, aliases=()):
    cols = list(columns)
    if exact and exact in cols:
        return exact
    by_key = {_header_key(c): c for c in cols}
    if exact and _header_key(exact) in by_key:
        return by_key[_header_key(exact)]
    for alias in aliases:
        k = _header_key(alias)
        if k in by_key:
            return by_key[k]
    return None


CARTHAGO_DOM_REF_ALIASES = [
    'Numéro de domiciliation', 'Numero de domiciliation', 'N° domiciliation',
    'N° de domiciliation', 'No domiciliation', 'Référence domiciliation',
    'Reference domiciliation', 'Réf domiciliation', 'Ref domiciliation',
    'Numero DOM', 'Numéro DOM', 'N° DOM',
]
CARTHAGO_TYPE_ALIASES = [
    'Type', 'Type dossier', 'Type de dossier', 'Nature dossier',
    'DOM/PREDOM', 'Type DOM/PREDOM', 'Type domiciliation',
]


def classify_carthago_type(value, sheet_name=None):
    txt = _text_key(value)
    sh = _text_key(sheet_name)
    combined = f'{txt} {sh}'.strip()
    if 'PREDOM' in combined or 'PRE DOM' in combined or 'PRE-DOM' in str(value or '').upper():
        return 'PREDOM'
    # N'inférer DOM depuis le nom de feuille que pour des noms explicites.
    if txt:
        if txt == 'DOM' or 'DOMICILIATION' in txt:
            return 'DOM'
    if sh in {'DOM', 'DOMICILIATION', 'DOMICILIATIONS'}:
        return 'DOM'
    return 'UNKNOWN'


def load_carthago_reference(path):
    meta = {
        'available': False,
        'path': str(path),
        'client_col': None,
        'name_col': None,
        'dom_ref_col': None,
        'type_col': None,
        'message': None,
    }
    if not Path(path).exists():
        meta['message'] = 'Fichier CarthagoDom absent; aucune proposition Carthago ne sera faite.'
        return pd.DataFrame(), meta

    xls = pd.ExcelFile(path)
    frames = []
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet, dtype=str)
        if df is None or df.empty:
            continue
        df = df.dropna(how='all').copy()
        if df.empty:
            continue
        df['_SOURCE_SHEET'] = sheet
        frames.append(df)

    if not frames:
        meta['message'] = 'Classeur CarthagoDom sans données exploitables.'
        return pd.DataFrame(), meta

    df = pd.concat(frames, ignore_index=True, sort=False)
    client_col = detect_column(df.columns, CARTHAGO_CLIENT_COL, ['Client'])
    name_col = detect_column(df.columns, CARTHAGO_NAME_COL, ['Nom complet/Raison sociale'])
    dom_ref_col = detect_column(df.columns, CARTHAGO_DOM_REF_COL, CARTHAGO_DOM_REF_ALIASES)
    type_col = detect_column(df.columns, CARTHAGO_TYPE_COL, CARTHAGO_TYPE_ALIASES)

    if not client_col:
        raise ValueError('Colonne Carthago Client introuvable. Renseigner CARTHAGO_CLIENT_COL.')
    if not name_col:
        raise ValueError('Colonne Carthago Nom complet/Raison sociale introuvable. Renseigner CARTHAGO_NAME_COL.')

    df['_CARTHAGO_CLIENT_NORM'] = df[client_col].map(normalize_carthago_client)
    df['_CARTHAGO_NAME_NORM'] = df[name_col].map(normalize_name_for_match)

    if dom_ref_col:
        # La colonne Carthago peut être déjà compacte ou contenir des séparateurs.
        # Pour le matching, on applique la même clé que NUMERO_DOMICILIATION_VALIDE :
        # uniquement les chiffres + suffixe DZD.
        df['_CARTHAGO_DOM_REF_NORM'] = df[dom_ref_col].map(normalize_domiciliation_compare_key)
    else:
        df['_CARTHAGO_DOM_REF_NORM'] = None

    df['_CARTHAGO_TYPE'] = [
        classify_carthago_type(
            row.get(type_col) if type_col else None,
            row.get('_SOURCE_SHEET')
        )
        for _, row in df.iterrows()
    ]
    df['_CARTHAGO_ROW_ID'] = range(1, len(df) + 1)

    meta.update({
        'available': True,
        'client_col': client_col,
        'name_col': name_col,
        'dom_ref_col': dom_ref_col,
        'type_col': type_col,
        'message': 'OK' if dom_ref_col else 'Colonne numéro DOM non détectée; propositions désactivées jusqu’au paramétrage.',
    })
    return df, meta


def _record_value(processed, doc_type, field):
    """Retourne la première valeur normalisée comparable + contexte de page/trace."""
    for rec in processed.get('page_records') or []:
        if rec.get('doc_type') != doc_type:
            continue
        v = (rec.get('normalized_data') or {}).get(field)
        tr = (rec.get('normalization_trace') or {}).get(field) or {}
        raw = (rec.get('raw_data') or {}).get(field)
        if comparable(v) or comparable(raw):
            return {
                'value': v,
                'raw': raw,
                'page': rec.get('page_num'),
                'trace': tr,
            }
    return {'value': None, 'raw': None, 'page': None, 'trace': {}}


def match_domiciliation_carthago(processed, carthago_df, meta):
    """
    V5.5 — ordre métier strict :
      1) contrôler DOM_COMPTE_LOCAL et extraire SIEGE_RACINE = 07310 + 6 chiffres ;
      2) filtrer Carthago sur Client == SIEGE_RACINE ;
      3) afficher la similarité nom à titre informationnel ;
      4) comparer exactement NUMERO_DOMICILIATION_VALIDE normalisé avec
         "Numero de domiciliation" Carthago normalisé ;
      5) proposer uniquement le numéro de domiciliation qui matche exactement.

    Le contrôle Modulo 97 du compte est séparé du matching Carthago et ne modifie
    jamais automatiquement le compte extrait.
    """
    compte = _record_value(processed, 'ENGAGEMENT_DOMICILIATION', 'DOM_COMPTE_LOCAL')
    ctr_name = _record_value(processed, 'CONTRAT_TRAVAIL', 'CTR_NOM_PRENOM_TRAVAILLEUR')
    ctr_ref = _record_value(processed, 'CONTRAT_TRAVAIL', 'CTR_REFERENCE_DOMICILIATION')

    compte_value = compte.get('value') or compte.get('raw')
    compte_info = analyze_dom_compte_local(compte_value)
    siege = compte_info.get('SIEGE_RACINE')
    ctr_ref_norm = ctr_ref.get('value')
    ctr_ref_raw = ctr_ref.get('raw')
    ctr_ref_compare_key = normalize_domiciliation_compare_key(ctr_ref_norm or ctr_ref_raw)

    summary = {
        **compte_info,
        'DOM_COMPTE_LOCAL': compte_value,
        'DOM_COMPTE_LOCAL_PAGE': compte.get('page'),
        'CTR_NOM_PRENOM_TRAVAILLEUR': ctr_name.get('value') or ctr_name.get('raw'),
        'CTR_REFERENCE_DOMICILIATION_RAW': ctr_ref_raw,
        'CTR_REFERENCE_DOMICILIATION_NORMALIZED': ctr_ref_norm,
        'NUMERO_DOMICILIATION_NORMALIZED': ctr_ref_compare_key,
        'CTR_REFERENCE_DOMICILIATION_PAGE': ctr_ref.get('page'),
        'CTR_REFERENCE_DOMICILIATION_NORMALIZATION_STATUS': (ctr_ref.get('trace') or {}).get('status'),
        'CARTHAGO_MATCH_STATUS': None,
        'CARTHAGO_CLIENT_MATCH': False,
        'CARTHAGO_CLIENT': None,
        'CARTHAGO_NOM_COMPLET': None,
        'SIMILARITE_NOM': None,
        'CONTROLE_NOM': None,
        'CARTHAGO_TYPE': None,
        'NUMERO_DOMICILIATION_PROPOSE': None,
        'CARTHAGO_NUM_DOM_MATCH_RAW': None,
        'CARTHAGO_NUM_DOM_MATCH_NORMALIZED': None,
        'CARTHAGO_CANDIDATS_DOM': None,
        'CARTHAGO_CANDIDATS_PREDOM': None,
        'REGLE_PROPOSITION_DOM': None,
        'CARTHAGO_SOURCE_SHEET': None,
    }
    details = []

    if not meta.get('available'):
        summary['CARTHAGO_MATCH_STATUS'] = 'CARTHAGO_FILE_MISSING'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_REFERENCE_FILE_MISSING'
        summary['DOM_COMPTE_LOCAL_REQUIRES_MANUAL'] = True
        return summary, details

    if not siege:
        summary['CARTHAGO_MATCH_STATUS'] = 'SIEGE_RACINE_NOT_FOUND'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_SIEGE_RACINE_MISSING'
        summary['DOM_COMPTE_LOCAL_REQUIRES_MANUAL'] = True
        return summary, details

    candidates = carthago_df[carthago_df['_CARTHAGO_CLIENT_NORM'] == siege].copy()
    if candidates.empty:
        summary['CARTHAGO_MATCH_STATUS'] = 'CLIENT_NOT_FOUND'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_CLIENT_NOT_FOUND'
        summary['DOM_COMPTE_LOCAL_STATUS'] = (
            str(summary.get('DOM_COMPTE_LOCAL_STATUS') or '') + '|CLIENT_CARTHAGO_NOT_FOUND'
        ).strip('|')
        summary['DOM_COMPTE_LOCAL_REQUIRES_MANUAL'] = True
        return summary, details

    summary['CARTHAGO_CLIENT_MATCH'] = True
    client_col = meta['client_col']
    name_col = meta['name_col']
    dom_col = meta['dom_ref_col']

    if not dom_col:
        summary['CARTHAGO_MATCH_STATUS'] = 'CARTHAGO_DOM_COLUMN_NOT_FOUND'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_NUMERO_DOMICILIATION_COLUMN_MISSING'
        return summary, details

    rows = []
    for _, r in candidates.iterrows():
        sim = name_similarity(
            ctr_name.get('value') or ctr_name.get('raw'),
            r.get(name_col)
        )
        dom_raw = r.get(dom_col)
        dom_norm = r.get('_CARTHAGO_DOM_REF_NORM')
        exact_ref = bool(ctr_ref_compare_key and dom_norm and ctr_ref_compare_key == dom_norm)

        item = {
            'FICHIER': processed.get('source_file'),
            'DOM_COMPTE_LOCAL': compte_value,
            'DOM_COMPTE_LOCAL_NORMALIZED': compte_info.get('DOM_COMPTE_LOCAL_NORMALIZED'),
            'COMPTE_LONGUEUR': compte_info.get('COMPTE_LONGUEUR'),
            'PREFIXE_02700_OK': compte_info.get('PREFIXE_02700_OK'),
            'SIEGE_RACINE': siege,
            'RACINE_6': compte_info.get('RACINE_6'),
            'ORDINAL_APRES_RACINE': compte_info.get('ORDINAL_APRES_RACINE'),
            'ORDINAL_001_OK': compte_info.get('ORDINAL_001_OK'),
            'CLE_COMPTE': compte_info.get('CLE_COMPTE'),
            'CLE_ATTENDUE_MOD97': compte_info.get('CLE_ATTENDUE_MOD97'),
            'MODULO97_RESTE': compte_info.get('MODULO97_RESTE'),
            'MODULO97_OK': compte_info.get('MODULO97_OK'),
            'DOM_COMPTE_LOCAL_STATUS': compte_info.get('DOM_COMPTE_LOCAL_STATUS'),
            'CARTHAGO_CLIENT_MATCH': True,
            'CARTHAGO_ROW_ID': r.get('_CARTHAGO_ROW_ID'),
            'CARTHAGO_SOURCE_SHEET': r.get('_SOURCE_SHEET'),
            'CARTHAGO_CLIENT': r.get(client_col),
            'CTR_NOM_PRENOM_TRAVAILLEUR': ctr_name.get('value') or ctr_name.get('raw'),
            'CARTHAGO_NOM_COMPLET': r.get(name_col),
            'SIMILARITE_NOM': sim,
            'CARTHAGO_TYPE': r.get('_CARTHAGO_TYPE'),
            'CARTHAGO_NUM_DOM_RAW': dom_raw,
            'CARTHAGO_NUM_DOM_NORMALIZED': dom_norm,
            'CTR_REFERENCE_DOMICILIATION_NORMALIZED': ctr_ref_norm,
            'NUMERO_DOMICILIATION_NORMALIZED': ctr_ref_compare_key,
            'REFERENCE_EXACT_MATCH': exact_ref,
        }
        details.append(item)
        rows.append((r, item))

    best_name_item = max(
        (x[1] for x in rows if x[1].get('SIMILARITE_NOM') is not None),
        key=lambda z: z['SIMILARITE_NOM'],
        default=rows[0][1] if rows else None
    )
    if best_name_item:
        summary['CARTHAGO_CLIENT'] = best_name_item.get('CARTHAGO_CLIENT')
        summary['CARTHAGO_NOM_COMPLET'] = best_name_item.get('CARTHAGO_NOM_COMPLET')
        summary['SIMILARITE_NOM'] = best_name_item.get('SIMILARITE_NOM')
        sim = best_name_item.get('SIMILARITE_NOM')
        if sim is None:
            summary['CONTROLE_NOM'] = 'NON_CALCULABLE'
        elif sim >= 90:
            summary['CONTROLE_NOM'] = 'SIMILARITE_FORTE'
        elif sim >= 75:
            summary['CONTROLE_NOM'] = 'SIMILARITE_MOYENNE'
        else:
            summary['CONTROLE_NOM'] = 'SIMILARITE_FAIBLE'

    # Le nom reste informatif : il ne change jamais CARTHAGO_CLIENT_MATCH.
    if summary.get('DOM_COMPTE_LOCAL_REQUIRES_MANUAL') is False and summary.get('CARTHAGO_CLIENT_MATCH'):
        # Structure + modulo valides + Client Carthago exact => pas de validation manuelle du compte.
        summary['DOM_COMPTE_LOCAL_STATUS'] = 'COMPTE_OK_CLIENT_CARTHAGO_OK'

    def uniq_refs(items):
        vals = []
        for _, item in items:
            v = item.get('CARTHAGO_NUM_DOM_NORMALIZED') or item.get('CARTHAGO_NUM_DOM_RAW')
            if comparable(v) and str(v) not in vals:
                vals.append(str(v))
        return vals

    dom_rows = [x for x in rows if x[1].get('CARTHAGO_TYPE') == 'DOM']
    predom_rows = [x for x in rows if x[1].get('CARTHAGO_TYPE') == 'PREDOM']
    dom_refs = uniq_refs(dom_rows)
    predom_refs = uniq_refs(predom_rows)
    summary['CARTHAGO_CANDIDATS_DOM'] = ' | '.join(dom_refs) if dom_refs else None
    summary['CARTHAGO_CANDIDATS_PREDOM'] = ' | '.join(predom_refs) if predom_refs else None

    if not ctr_ref_compare_key:
        summary['CARTHAGO_MATCH_STATUS'] = 'NUMERO_DOMICILIATION_NOT_NORMALIZABLE'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_NUMERO_DOMICILIATION_MISSING_OR_INVALID'
        return summary, details

    exact_rows = [x for x in rows if x[1].get('REFERENCE_EXACT_MATCH')]
    if not exact_rows:
        summary['CARTHAGO_MATCH_STATUS'] = 'NO_EXACT_DOMICILIATION_REFERENCE_MATCH'
        summary['REGLE_PROPOSITION_DOM'] = (
            'CLIENT_EXACT + NAME_SIMILARITY_INFORMATIONAL + '
            'NO_EXACT_MATCH(NUMERO_DOMICILIATION_NORMALIZED, CARTHAGO_NUMERO_DOMICILIATION)'
        )
        return summary, details

    exact_values = []
    for _, item in exact_rows:
        v = item.get('CARTHAGO_NUM_DOM_NORMALIZED')
        if comparable(v) and str(v) not in exact_values:
            exact_values.append(str(v))

    if len(exact_values) != 1:
        summary['CARTHAGO_MATCH_STATUS'] = 'MULTIPLE_EXACT_REFERENCE_VALUES'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_MULTIPLE_DISTINCT_EXACT_REFERENCES'
        return summary, details

    exact_types = {x[1].get('CARTHAGO_TYPE') for x in exact_rows}
    if exact_types == {'PREDOM'}:
        chosen = exact_rows[0][1]
        summary['CARTHAGO_CLIENT'] = chosen.get('CARTHAGO_CLIENT')
        summary['CARTHAGO_NOM_COMPLET'] = chosen.get('CARTHAGO_NOM_COMPLET')
        summary['SIMILARITE_NOM'] = chosen.get('SIMILARITE_NOM')
        summary['CARTHAGO_TYPE'] = 'PREDOM'
        summary['CARTHAGO_NUM_DOM_MATCH_RAW'] = chosen.get('CARTHAGO_NUM_DOM_RAW')
        summary['CARTHAGO_NUM_DOM_MATCH_NORMALIZED'] = chosen.get('CARTHAGO_NUM_DOM_NORMALIZED')
        summary['CARTHAGO_SOURCE_SHEET'] = chosen.get('CARTHAGO_SOURCE_SHEET')
        summary['CARTHAGO_MATCH_STATUS'] = 'EXACT_REFERENCE_PREDOM_ONLY'
        summary['REGLE_PROPOSITION_DOM'] = 'NO_PROPOSAL_EXACT_REFERENCE_FOUND_ONLY_AS_PREDOM'
        return summary, details

    chosen_pair = next((x for x in exact_rows if x[1].get('CARTHAGO_TYPE') == 'DOM'), exact_rows[0])
    chosen = chosen_pair[1]
    proposed = exact_values[0]

    summary['NUMERO_DOMICILIATION_PROPOSE'] = proposed
    summary['CARTHAGO_TYPE'] = chosen.get('CARTHAGO_TYPE')
    summary['CARTHAGO_CLIENT'] = chosen.get('CARTHAGO_CLIENT')
    summary['CARTHAGO_NOM_COMPLET'] = chosen.get('CARTHAGO_NOM_COMPLET')
    summary['SIMILARITE_NOM'] = chosen.get('SIMILARITE_NOM')
    summary['CARTHAGO_NUM_DOM_MATCH_RAW'] = chosen.get('CARTHAGO_NUM_DOM_RAW')
    summary['CARTHAGO_NUM_DOM_MATCH_NORMALIZED'] = chosen.get('CARTHAGO_NUM_DOM_NORMALIZED')
    summary['CARTHAGO_SOURCE_SHEET'] = chosen.get('CARTHAGO_SOURCE_SHEET')

    if chosen.get('CARTHAGO_TYPE') == 'DOM':
        summary['CARTHAGO_MATCH_STATUS'] = 'EXACT_DOMICILIATION_REFERENCE_MATCH'
        summary['REGLE_PROPOSITION_DOM'] = (
            'CLIENT_EXACT + NAME_SIMILARITY_INFORMATIONAL + '
            'EXACT_NORMALIZED_DOMICILIATION_REFERENCE_MATCH + DOM'
        )
    else:
        summary['CARTHAGO_MATCH_STATUS'] = 'EXACT_DOMICILIATION_REFERENCE_MATCH_TYPE_UNKNOWN'
        summary['REGLE_PROPOSITION_DOM'] = (
            'CLIENT_EXACT + NAME_SIMILARITY_INFORMATIONAL + '
            'EXACT_NORMALIZED_DOMICILIATION_REFERENCE_MATCH; TYPE_NOT_REQUIRED_FOR_MATCH'
        )

    return summary, details

def build_domiciliation_manual_row(processed, match_summary, carthago_meta):
    """
    Une ligne par dossier, même si le numéro OCR semble correct.
    La Partie 2B exigera VALIDEE / CORRIGEE / REJETEE avant certification.
    """
    dq = processed.get('dq_summary') or {}
    ctx = processed.get('business_context') or {}
    proposal = match_summary.get('NUMERO_DOMICILIATION_PROPOSE')
    return {
        'FICHIER': processed.get('source_file'),
        'TYPE_DOSSIER': ctx.get('TYPE_DOSSIER'),
        'DOSSIER_STATUS': dq.get('VALIDATION_AUTO'),
        'DQ_SCORE': dq.get('DQ_SCORE'),
        'SEVERITE': 'CONTROLE_METIER',
        'PAGE': match_summary.get('CTR_REFERENCE_DOMICILIATION_PAGE'),
        'TYPE_DOCUMENT': 'CONTRAT_TRAVAIL',
        'CHAMP': 'NUMERO_DOMICILIATION_VALIDE',
        'RAW': match_summary.get('CTR_REFERENCE_DOMICILIATION_RAW'),
        # Pour ce champ synthétique, NORMALIZED correspond au format Carthago
        # compact : chiffres uniquement + DZD.
        'NORMALIZED': match_summary.get('NUMERO_DOMICILIATION_NORMALIZED'),
        'CTR_REFERENCE_DOMICILIATION_NORMALIZED': match_summary.get('CTR_REFERENCE_DOMICILIATION_NORMALIZED'),
        'NORMALIZATION_STATUS': match_summary.get('CTR_REFERENCE_DOMICILIATION_NORMALIZATION_STATUS'),
        'FIELD_CONFIDENCE': None,
        'TYPE_ANOMALIE': 'VALIDATION_NUMERO_DOMICILIATION',
        'MOTIF': match_summary.get('CARTHAGO_MATCH_STATUS'),
        'VALEUR_PROPOSEE': proposal,
        'SOURCE_PROPOSITION_DOCUMENT': 'CARTHAGO_DOM' if proposal else None,
        'SOURCE_PROPOSITION_PAGE': match_summary.get('CARTHAGO_SOURCE_SHEET'),
        'SOURCE_PROPOSITION_CHAMP': carthago_meta.get('dom_ref_col'),
        'MEME_CLIENT_STATUT': 'CLIENT_EXACT' if match_summary.get('CARTHAGO_CLIENT') else match_summary.get('CARTHAGO_MATCH_STATUS'),
        'MEME_CLIENT_PREUVE': (
            f"SIEGE_RACINE={match_summary.get('SIEGE_RACINE')} == Client={match_summary.get('CARTHAGO_CLIENT')}"
            if match_summary.get('CARTHAGO_CLIENT') else None
        ),
        'REGLE_PROPOSITION': match_summary.get('REGLE_PROPOSITION_DOM'),
        'SIEGE_RACINE': match_summary.get('SIEGE_RACINE'),
        'CARTHAGO_CLIENT': match_summary.get('CARTHAGO_CLIENT'),
        'CTR_NOM_PRENOM_TRAVAILLEUR': match_summary.get('CTR_NOM_PRENOM_TRAVAILLEUR'),
        'CARTHAGO_NOM_COMPLET': match_summary.get('CARTHAGO_NOM_COMPLET'),
        'SIMILARITE_NOM': match_summary.get('SIMILARITE_NOM'),
        'CONTROLE_NOM': match_summary.get('CONTROLE_NOM'),
        'CARTHAGO_TYPE': match_summary.get('CARTHAGO_TYPE'),
        'CARTHAGO_NUM_DOM_MATCH_RAW': match_summary.get('CARTHAGO_NUM_DOM_MATCH_RAW'),
        'CARTHAGO_NUM_DOM_MATCH_NORMALIZED': match_summary.get('CARTHAGO_NUM_DOM_MATCH_NORMALIZED'),
        'CARTHAGO_MATCH_STATUS': match_summary.get('CARTHAGO_MATCH_STATUS'),
        'CARTHAGO_CANDIDATS_DOM': match_summary.get('CARTHAGO_CANDIDATS_DOM'),
        'CARTHAGO_CANDIDATS_PREDOM': match_summary.get('CARTHAGO_CANDIDATS_PREDOM'),
        'VALEUR_VALIDEE': None,
        'DECISION_MANUELLE': None,
        'COMMENTAIRE': None,
    }


def build_compte_local_manual_row(processed, match_summary):
    """Ajoute le compte dans la même validation humaine seulement s'il faut le revoir."""
    if not match_summary.get('DOM_COMPTE_LOCAL_REQUIRES_MANUAL'):
        return None
    dq = processed.get('dq_summary') or {}
    ctx = processed.get('business_context') or {}
    motifs = []
    if not match_summary.get('COMPTE_20_CHIFFRES'):
        motifs.append('LONGUEUR_DIFFERENTE_DE_20')
    if not match_summary.get('PREFIXE_02700_OK'):
        motifs.append('PREFIXE_02700_INVALIDE')
    if not match_summary.get('SIEGE_RACINE'):
        motifs.append('SIEGE_RACINE_07310_PLUS_6_INTRouvable')
    if not match_summary.get('ORDINAL_001_OK'):
        motifs.append('ORDINAL_001_INVALIDE')
    if match_summary.get('MODULO97_OK') is False:
        motifs.append('CLE_MODULO97_INVALIDE')
    if match_summary.get('SIEGE_RACINE') and not match_summary.get('CARTHAGO_CLIENT_MATCH'):
        motifs.append('CLIENT_CARTHAGO_NON_TROUVE')

    return {
        'FICHIER': processed.get('source_file'),
        'TYPE_DOSSIER': ctx.get('TYPE_DOSSIER'),
        'DOSSIER_STATUS': dq.get('VALIDATION_AUTO'),
        'DQ_SCORE': dq.get('DQ_SCORE'),
        'SEVERITE': 'CONTROLE_METIER',
        'PAGE': match_summary.get('DOM_COMPTE_LOCAL_PAGE'),
        'TYPE_DOCUMENT': 'ENGAGEMENT_DOMICILIATION',
        'CHAMP': 'DOM_COMPTE_LOCAL',
        'RAW': match_summary.get('DOM_COMPTE_LOCAL'),
        'NORMALIZED': match_summary.get('DOM_COMPTE_LOCAL_NORMALIZED'),
        'NORMALIZATION_STATUS': 'REVIEW',
        'FIELD_CONFIDENCE': None,
        'TYPE_ANOMALIE': 'VALIDATION_COMPTE_LOCAL',
        'MOTIF': ' | '.join(motifs) if motifs else match_summary.get('DOM_COMPTE_LOCAL_STATUS'),
        # Pas de reconstruction automatique du compte complet : le Client Carthago ne contient
        # que SIEGE_RACINE et la clé Modulo 97 attendue reste informative.
        'VALEUR_PROPOSEE': None,
        'SOURCE_PROPOSITION_DOCUMENT': None,
        'SOURCE_PROPOSITION_PAGE': None,
        'SOURCE_PROPOSITION_CHAMP': None,
        'MEME_CLIENT_STATUT': 'CLIENT_EXACT' if match_summary.get('CARTHAGO_CLIENT_MATCH') else 'A_VERIFIER',
        'MEME_CLIENT_PREUVE': (
            f"SIEGE_RACINE={match_summary.get('SIEGE_RACINE')} == Client={match_summary.get('CARTHAGO_CLIENT')}"
            if match_summary.get('CARTHAGO_CLIENT_MATCH') else None
        ),
        'REGLE_PROPOSITION': 'AUCUNE_CORRECTION_AUTOMATIQUE_DU_COMPTE',
        'SIEGE_RACINE': match_summary.get('SIEGE_RACINE'),
        'RACINE_6': match_summary.get('RACINE_6'),
        'CARTHAGO_CLIENT': match_summary.get('CARTHAGO_CLIENT'),
        'CTR_NOM_PRENOM_TRAVAILLEUR': match_summary.get('CTR_NOM_PRENOM_TRAVAILLEUR'),
        'CARTHAGO_NOM_COMPLET': match_summary.get('CARTHAGO_NOM_COMPLET'),
        'SIMILARITE_NOM': match_summary.get('SIMILARITE_NOM'),
        'CONTROLE_NOM': match_summary.get('CONTROLE_NOM'),
        'PREFIXE_02700_OK': match_summary.get('PREFIXE_02700_OK'),
        'ORDINAL_APRES_RACINE': match_summary.get('ORDINAL_APRES_RACINE'),
        'ORDINAL_001_OK': match_summary.get('ORDINAL_001_OK'),
        'CLE_COMPTE': match_summary.get('CLE_COMPTE'),
        'CLE_ATTENDUE_MOD97': match_summary.get('CLE_ATTENDUE_MOD97'),
        'MODULO97_RESTE': match_summary.get('MODULO97_RESTE'),
        'MODULO97_OK': match_summary.get('MODULO97_OK'),
        'DOM_COMPTE_LOCAL_STATUS': match_summary.get('DOM_COMPTE_LOCAL_STATUS'),
        'VALEUR_VALIDEE': None,
        'DECISION_MANUELLE': None,
        'COMMENTAIRE': None,
    }


# Tests V5.3 demandés.
assert extract_siege_racine('02700731010910800156') == '07310109108'
_test_compte = analyze_dom_compte_local('02700731010910800156')
assert _test_compte['RACINE_6'] == '109108'
assert _test_compte['ORDINAL_APRES_RACINE'] == '001'
assert _test_compte['CLE_COMPTE'] == '56'
assert _test_compte['MODULO97_RESTE'] == 62
assert _test_compte['CLE_ATTENDUE_MOD97'] == '91'
assert _test_compte['MODULO97_OK'] is False
_test_rib_ok = analyze_dom_compte_local('02700712000001400194')
assert _test_rib_ok['MODULO97_OK'] is True
assert _test_rib_ok['CLE_ATTENDUE_MOD97'] == '94'
assert name_similarity('YILDIRIM IBRAHIM', 'IBRAHIM YILDIRIM') >= 99
assert normalize_domiciliation_compare_key('271901|2026.1|40|001345|DZD') == '2719012026140001345DZD'
print('✅ SIEGE_RACINE regex : 02700731010910800156 ->', extract_siege_racine('02700731010910800156'))
print('✅ Contrôle compte exemple : reste modulo97=', _test_compte['MODULO97_RESTE'], '| clé lue=', _test_compte['CLE_COMPTE'], '| clé attendue=', _test_compte['CLE_ATTENDUE_MOD97'])
print('✅ Similarité nom : contrôle informationnel après match SIEGE_RACINE == Client')
print('✅ Clé DOM Carthago : 271901|2026.1|40|001345|DZD -> 2719012026140001345DZD')


### V5.5 — garde-fous anti-faux-positifs

Les signaux historiques de la Partie 1 sont réévalués à partir des valeurs courantes de la Partie 2. Un ancien `critical_fields_missing`, `PARTIELLE` ou `LOW_FILL_RATE` ne déclenche plus une validation si le champ est maintenant présent et normalisé. Les écarts attendus sur un dossier `AUGMENTATION` ne sont pas traités comme des conflits de contrat initial.


In [ ]:
# Tests de non-régression V5.5
_ctx_new={'TYPE_DOSSIER':'NOUVEAU_CONTRAT'}
_ctx_aug={'TYPE_DOSSIER':'AUGMENTATION'}
_r={
    'doc_type':'TITRE_TRAVAIL','page_num':6,
    'raw_data':{'TTR_DATE_FIN':'23/10/2025'},
    'normalized_data':{'TTR_DATE_FIN':'23/10/2025'},
    'normalization_trace':{'TTR_DATE_FIN':{'status':'RAW_OK'}},
    'critical_fields_missing':['TTR_DATE_FIN'],
    'extraction_status':'PARTIELLE','classification_review_required':False,
    'quality_flags':['CRITICAL_MISSING_OR_STRUCTURALLY_SUSPECT','LOW_FILL_RATE'],
}
assert 'TTR_DATE_FIN' not in relevant_critical_missing(_r,_ctx_new)
# Les autres champs critiques du TTR ne sont pas renseignés dans ce mini-test;
# on vérifie seulement que TTR_DATE_FIN n'est plus considéré manquant.
assert 'EXTRACTION_PARTIELLE' not in page_regulatory_flags(_r,_ctx_new)
_r2=dict(_r)
_r2['raw_data']={'TTR_DATE_FIN':None}
_r2['normalized_data']={'TTR_DATE_FIN':None}
_r2['normalization_trace']={'TTR_DATE_FIN':{'status':'MISSING'}}
assert 'TTR_DATE_FIN' in relevant_critical_missing(_r2,_ctx_new)
assert 'CRITICAL_FIELD_MISSING' in page_regulatory_flags(_r2,_ctx_new)
for _g in CROSS_DOCUMENT_GROUPS:
    if _g['name'] in {'SALAIRE_NET','PART_TRANSFERABLE','DATE_DEBUT_CONTRAT'}:
        assert not crosscheck_group_applicable(_g,_ctx_aug)
    elif _g['name'] in {'NUMERO_PERMIS','DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'}:
        assert crosscheck_group_applicable(_g,_ctx_aug)
print('✅ Tests anti-faux-positifs V5.5 : OK')


## 5B. V6 — Exploitation de la confidence V9.3.1 et simplification de la validation

La confidence est combinée avec normalisation, cross-check et cohérence métier. Elle ne constitue pas une probabilité native Qwen.


In [ ]:

# =====================================================================
# V6 — OVERRIDES V9.3.1 : confidence, page absente, durée, UX validation
# =====================================================================

# Garder les fonctions V5.5 disponibles pour audit/régression.
_normalize_amount_trace_v55 = normalize_amount_trace
_calculate_dq_summary_v55 = calculate_dq_summary


def normalize_amount_trace(raw):
    """
    V6 : même normalisation conservatrice que V5.5 + cas d'impression ':'.
    Sur les montants, ':' est traité comme un séparateur visuel équivalent à '.'
    uniquement si la chaîne ne contient que chiffres/séparateurs autorisés.
    Exemples :
      454:835:67    -> 454835.67
      47:719:857,6  -> 47719857.60
      23:340.43     -> 23340.43
    Aucune conversion OCR lettre->chiffre.
    """
    if raw is None:
        return _normalize_amount_trace_v55(raw)

    s=str(raw).replace('\u00a0',' ').strip()
    # Ne toucher au ':' que dans une expression de montant pure.
    if ':' in s:
        upper=s.upper().replace('DZD','').replace('DA','').replace('EUR','').replace('€','').strip()
        candidate=upper.lstrip('+-').strip()
        if re.fullmatch(r"[0-9:., '\u2019]+", candidate or ''):
            transformed=s.replace(':','.')
            tr=_normalize_amount_trace_v55(transformed)
            if tr.get('status') in {'RAW_OK','AUTO_OK'}:
                tr=dict(tr)
                tr['raw']=raw
                tr['changed']=tr.get('normalized') != raw
                tr['rule']='AMOUNT_COLON_AS_DOT__' + str(tr.get('rule'))
                tr['message']="':' interprété comme séparateur d'impression de montant"
                return tr
    return _normalize_amount_trace_v55(raw)


def _source_field_confidence(rec, field):
    fc=(rec.get('field_confidence') or {}).get(field) or {}
    try:
        score=float(fc.get('score')) if fc.get('score') is not None else None
    except Exception:
        score=None
    return {
        'score':score,
        'band':fc.get('band'),
        'basis':fc.get('basis'),
        'attempts_supporting':fc.get('attempts_supporting'),
        'distinct_valid_values':fc.get('distinct_valid_values'),
        'page_score':rec.get('extraction_confidence_score'),
        'page_band':rec.get('extraction_confidence_band'),
        'recovery':bool(rec.get('page_recovery_triggered')),
        'recovery_passes':rec.get('recovery_passes'),
    }


def field_evidence(processed_records, field):
    """V6 : evidence V5.5 + confidence native opérationnelle V9.3.1."""
    ev=[]
    for r in processed_records:
        nd=r.get('normalized_data') or {}
        tr=r.get('normalization_trace') or {}
        if field in nd:
            src=_source_field_confidence(r,field)
            ev.append({
                'doc_type':r.get('doc_type'),
                'page':r.get('page_num'),
                'value':nd.get(field),
                'raw':(r.get('raw_data') or {}).get(field),
                'trace':tr.get(field) or {},
                'classification_review_required':bool(r.get('classification_review_required')),
                'quality_flags':penalizing_quality_flags(r),
                'extraction_status':r.get('extraction_status'),
                'source_confidence':src,
            })
    return ev


def _owner_doc_type(field):
    for dt,fields in FIELD_SCHEMA.items():
        if field in fields:
            return dt
    return None


def _doc_presence(processed_records):
    return {r.get('doc_type') for r in processed_records if r.get('doc_type')}


def _missing_page_context(processed_records, field):
    owner=_owner_doc_type(field)
    if owner and owner not in _doc_presence(processed_records):
        return f'PAGE_ABSENTE:{owner}'
    return None


def add_expected_page_absence_anomalies(processed_records, anomalies, source):
    """
    Une ligne synthétique par type de document cœur absent.
    Le but est d'expliquer le vide sans créer 15/28/32 lignes de validation.
    """
    present=_doc_presence(processed_records)
    for dt in sorted(EXPECTED_CORE_DOC_TYPES - present):
        critical=sorted(f for f in CRITICAL_FIELDS if _owner_doc_type(f)==dt)
        anomalies.append({
            'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':dt,
            'TYPE_ANOMALIE':'EXPECTED_PAGE_ABSENT',
            'SEVERITE':MISSING_EXPECTED_PAGE_SEVERITY,
            'CHAMP':'[PAGE_ABSENTE]',
            'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
            'MOTIF':(
                f'Page {dt} absente du dossier; champs critiques non disponibles: '
                + ', '.join(critical)
            )
        })


def _parse_duration_months(value):
    if value is None:
        return None
    if isinstance(value,(int,float)) and not isinstance(value,bool):
        return float(value)
    s=str(value).upper().replace(',', '.')
    # "2 ANS 0 JOURS", "24 MOIS", etc.
    my=re.search(r'\b(\d{1,2}(?:\.\d+)?)\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES)\b',s)
    mm=re.search(r'\b(\d{1,3}(?:\.\d+)?)\s*MOIS\b',s)
    if my:
        return float(my.group(1))*12.0
    if mm:
        return float(mm.group(1))
    # entier brut = mois pour les champs *_DUREE_MOIS
    if re.fullmatch(r'\s*\d{1,3}\s*',s):
        return float(s.strip())
    return None


def _duration_check(label, start, end, duration, tolerance_days=120):
    d1=parse_date_safe(start); d2=parse_date_safe(end)
    months=_parse_duration_months(duration)
    if not (d1 and d2 and months is not None):
        return None
    observed=(d2-d1).days
    expected=months*30.4375
    delta=abs(observed-expected)
    return {
        'label':label,'start':start,'end':end,'duration':duration,
        'observed_days':observed,'expected_days':round(expected,1),
        'delta_days':round(delta,1),'ok':delta <= tolerance_days,
    }


def _first_norm(records, field):
    for r in records:
        v=(r.get('normalized_data') or {}).get(field)
        if comparable(v):
            return v
    return None


def add_business_controls(processed, anomalies, business_context):
    """
    V6 : ordre des dates + cohérence durée.
    Contrôles de durée :
      - ENGAGEMENT DOM : dates contrat vs DOM_DUREE_CONTRAT_MOIS
      - TITRE TRAVAIL : dates permis vs TTR_DUREE
      - CTR/CTS permis : comparés à TTR_DUREE si le titre existe
    """
    records=processed['page_records']
    source=processed['source_file']

    values={}
    for r in records:
        for f,v in (r.get('normalized_data') or {}).items():
            if comparable(v) and f not in values:
                values[f]=v

    date_pairs=[
        ('DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','CONTRAT'),
        ('CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTR'),
        ('CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTS'),
        ('TTR_DATE_DEBUT','TTR_DATE_FIN','PERMIS_TTR'),
    ]
    for f1,f2,label in date_pairs:
        d1=parse_date_safe(values.get(f1)); d2=parse_date_safe(values.get(f2))
        if d1 and d2 and d2<d1:
            anomalies.append({
                'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':'MULTI',
                'TYPE_ANOMALIE':'INVALID_DATE_ORDER','SEVERITE':'BLOQUANT',
                'CHAMP':label,'VALEUR_RAW':None,
                'VALEUR_NORMALISEE':f'{values.get(f1)} > {values.get(f2)}',
                'MOTIF':'date fin antérieure à date début'
            })

    checks=[]
    checks.append(_duration_check(
        'DOM_CONTRAT',
        values.get('DOM_DATE_DEBUT_CONTRAT'),
        values.get('DOM_DATE_FIN_CONTRAT'),
        values.get('DOM_DUREE_CONTRAT_MOIS'),
    ))
    checks.append(_duration_check(
        'TTR_PERMIS',
        values.get('TTR_DATE_DEBUT'),
        values.get('TTR_DATE_FIN'),
        values.get('TTR_DUREE'),
    ))
    # La durée du TTR décrit le permis ; elle peut confirmer les dates CTR/CTS.
    checks.append(_duration_check(
        'CTR_PERMIS_VS_TTR_DUREE',
        values.get('CTR_DATE_DEBUT_VALIDITE_PERMIS'),
        values.get('CTR_DATE_FIN_VALIDITE_PERMIS'),
        values.get('TTR_DUREE'),
    ))
    checks.append(_duration_check(
        'CTS_PERMIS_VS_TTR_DUREE',
        values.get('CTS_DATE_DEBUT_VALIDITE_PERMIS'),
        values.get('CTS_DATE_FIN_VALIDITE_PERMIS'),
        values.get('TTR_DUREE'),
    ))

    processed['duration_checks']=[x for x in checks if x]
    for c in processed['duration_checks']:
        if not c['ok']:
            anomalies.append({
                'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':'MULTI',
                'TYPE_ANOMALIE':'DATE_DURATION_INCONSISTENT',
                'SEVERITE':'BLOQUANT' if c['label'] in {'DOM_CONTRAT','TTR_PERMIS'} else 'REVIEW',
                'CHAMP':c['label'],
                'VALEUR_RAW':None,
                'VALEUR_NORMALISEE':(
                    f"{c['start']} -> {c['end']} | durée={c['duration']} | "
                    f"observé={c['observed_days']}j | attendu≈{c['expected_days']}j"
                ),
                'MOTIF':'dates début/fin incompatibles avec la durée'
            })


def _crosscheck_context_for_field(processed_records, business_context, field):
    parts=[]
    for group in CROSS_DOCUMENT_GROUPS:
        if field not in set(group.get('fields',{}).values()):
            continue
        if not crosscheck_group_applicable(group,business_context):
            parts.append(f"{group['name']}:N/A_AUGMENTATION")
            continue
        vals=[]
        for dt,f in group['fields'].items():
            for r in processed_records:
                if r.get('doc_type')==dt:
                    v=(r.get('normalized_data') or {}).get(f)
                    if comparable(v):
                        vals.append((dt,f,r.get('page_num'),v))
        if len(vals)<2:
            parts.append(f"{group['name']}:NON_COMPARABLE({len(vals)} source)")
        else:
            base=vals[0][3]
            ok=all(same_value(group['kind'],base,x[3]) for x in vals[1:])
            detail=', '.join(f"{dt}@p{p}={v}" for dt,f,p,v in vals)
            parts.append(f"{group['name']}:{'COHERENT' if ok else 'CONFLIT'} [{detail}]")
    return ' ; '.join(parts) if parts else 'PAS_DE_CROSSCHECK_DEFINI'


def _duration_context_for_field(processed, field):
    related=[]
    for c in processed.get('duration_checks') or []:
        mapping={
            'DOM_CONTRAT': {'DOM_DUREE_CONTRAT_MOIS','DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT'},
            'TTR_PERMIS': {'TTR_DUREE','TTR_DATE_DEBUT','TTR_DATE_FIN'},
            'CTR_PERMIS_VS_TTR_DUREE': {'TTR_DUREE','CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS'},
            'CTS_PERMIS_VS_TTR_DUREE': {'TTR_DUREE','CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS'},
        }
        if field in mapping.get(c['label'],set()):
            related.append(
                f"{c['label']}:{'OK' if c['ok'] else 'ANOMALIE'} "
                f"({c['start']}→{c['end']}; durée={c['duration']}; Δ={c['delta_days']}j)"
            )
    return ' ; '.join(related)


def _field_part1_context(rec, field):
    src=_source_field_confidence(rec,field)
    return (
        f"P1_CONF={src.get('score')} {src.get('band') or ''}"
        f" ({src.get('basis') or 'N/A'}); "
        f"PAGE_CONF={src.get('page_score')} {src.get('page_band') or ''}; "
        f"RECOVERY={'OUI' if src.get('recovery') else 'NON'}"
        + (f"/{src.get('recovery_passes')}" if src.get('recovery_passes') is not None else '')
    )


def calculate_field_confidence(processed_records, business_context, anomalies):
    """
    DATA_CONFIDENCE V6 = confidence opérationnelle V9.3.1 + DQ Partie 2.
    Ce score qualifie la donnée ; ce n'est ni une probabilité Qwen ni un score statistique calibré.
    """
    rows=[]
    present_docs=_doc_presence(processed_records)

    for field in ALL_FIELDS:
        applicability,importance=field_policy(field,business_context)
        ev=field_evidence(processed_records,field)
        present=[e for e in ev if comparable(e['value'])]
        owner=_owner_doc_type(field)

        if importance=='INFO_ONLY':
            rows.append({
                'CHAMP':field,'APPLICABILITE':applicability,'DQ_IMPORTANCE':importance,
                'FIELD_CONFIDENCE':None,'FIELD_CONFIDENCE_LEVEL':'INFO_ONLY',
                'FIELD_CONFIDENCE_REASON':'HORS_SCORE_DQ',
                'SOURCE_CONFIDENCE':None,'SOURCE_CONFIDENCE_BASIS':None,
                'NB_OCCURRENCES':len(present),
            })
            continue

        if owner not in present_docs:
            score=0
            source_score=None
            source_basis='PAGE_ABSENTE'
            reasons=[f'PAGE_ABSENTE:{owner}']
        elif not present:
            score=0
            source_score=0
            source_basis='MISSING'
            reasons=['AUCUNE_VALEUR_NORMALISEE']
        else:
            source_scores=[
                e['source_confidence'].get('score') for e in present
                if e['source_confidence'].get('score') is not None
            ]
            source_score=round(sum(source_scores)/len(source_scores),1) if source_scores else 70.0
            source_basis=' | '.join(dict.fromkeys(
                str(e['source_confidence'].get('basis'))
                for e in present if e['source_confidence'].get('basis')
            ))
            score=float(source_score)
            reasons=[f'PART1={source_score}']

            statuses=[(e['trace'] or {}).get('status') for e in present]
            if any(s=='REVIEW' for s in statuses):
                score=min(score,40)
                reasons.append('NORMALISATION_REVIEW_CAP40')
            elif all(s in {'RAW_OK','AUTO_OK'} for s in statuses):
                score=min(100,score+2)
                reasons.append('NORMALISATION_DETERMINISTE:+2')

            if any(e['classification_review_required'] for e in present):
                score=min(score,35)
                reasons.append('CLASSIFICATION_REVIEW_CAP35')

            relevant_conflicts=[
                a for a in anomalies
                if a.get('TYPE_ANOMALIE')=='CROSS_DOCUMENT_CONFLICT'
                and field in CROSSCHECK_FIELD_MAP.get(str(a.get('CHAMP')),set())
            ]
            if relevant_conflicts:
                score=min(score,45)
                reasons.append('CROSS_DOCUMENT_CONFLICT_CAP45')
            else:
                # Bonus léger uniquement si une vraie concordance multi-document existe.
                cross_ctx=_crosscheck_context_for_field(processed_records,business_context,field)
                if ':COHERENT [' in cross_ctx:
                    score=min(100,score+5)
                    reasons.append('CROSS_DOCUMENT_CONFIRMED:+5')

            if any(
                a.get('TYPE_ANOMALIE') in {'INVALID_DATE_ORDER','DATE_DURATION_INCONSISTENT'}
                and anomaly_applies_to_field(a,field)
                for a in anomalies
            ):
                score=min(score,35)
                reasons.append('DATE_COHERENCE_ERROR_CAP35')

            # Confidence basse V9.3.1 sur champ critique = review, pas rejet automatique.
            if importance=='CRITICAL' and source_score is not None and source_score < CONFIDENCE_CRITICAL_REVIEW:
                reasons.append('PART1_CRITICAL_CONFIDENCE_LOW')

            score=max(0,min(100,int(round(score))))

        level='HIGH' if score>=CONFIDENCE_HIGH else (
            'MEDIUM' if score>=CONFIDENCE_MEDIUM else ('LOW' if score>0 else 'MISSING')
        )
        rows.append({
            'CHAMP':field,'APPLICABILITE':applicability,'DQ_IMPORTANCE':importance,
            'FIELD_CONFIDENCE':score,'FIELD_CONFIDENCE_LEVEL':level,
            'FIELD_CONFIDENCE_REASON':' | '.join(reasons),
            'SOURCE_CONFIDENCE':source_score,
            'SOURCE_CONFIDENCE_BASIS':source_basis,
            'NB_OCCURRENCES':len(present),
        })
    return rows


def add_part1_confidence_anomalies(processed_records, anomalies, business_context, source):
    """
    Ne crée une validation que pour un champ CRITIQUE avec confidence P1 < 70
    et valeur présente. 82/84 ne sont donc pas des faux positifs.
    """
    existing={(a.get('TYPE_ANOMALIE'),a.get('PAGE'),a.get('CHAMP')) for a in anomalies}
    for r in processed_records:
        for field in FIELD_SCHEMA.get(r.get('doc_type'),[]):
            if field not in CRITICAL_FIELDS or not is_dq_relevant_field(field,business_context):
                continue
            val=(r.get('normalized_data') or {}).get(field)
            if not comparable(val):
                continue
            src=_source_field_confidence(r,field)
            score=src.get('score')
            if score is None or score >= CONFIDENCE_CRITICAL_REVIEW:
                continue
            key=('PART1_CRITICAL_CONFIDENCE_LOW',r.get('page_num'),field)
            if key in existing:
                continue
            anomalies.append({
                'FICHIER':source,'PAGE':r.get('page_num'),'TYPE_DOCUMENT':r.get('doc_type'),
                'TYPE_ANOMALIE':'PART1_CRITICAL_CONFIDENCE_LOW','SEVERITE':'REVIEW',
                'CHAMP':field,'VALEUR_RAW':(r.get('raw_data') or {}).get(field),
                'VALEUR_NORMALISEE':val,
                'MOTIF':(
                    f"confidence Partie1={score}; basis={src.get('basis')}; "
                    f"page_conf={src.get('page_score')}"
                )
            })


def anomaly_applies_to_field(anomaly, field, page=None, doc_type=None):
    typ=anomaly.get('TYPE_ANOMALIE')
    achamp=anomaly.get('CHAMP')
    apage=anomaly.get('PAGE')

    if typ in {'FORMAT_REVIEW','PART1_CRITICAL_CONFIDENCE_LOW'}:
        return achamp==field and (apage is None or page==apage)

    if typ in {'CRITICAL_FIELD_MISSING','CRITICAL_FIELD_MISSING_POSTPROCESS'}:
        parts={x.strip() for x in str(achamp or '').split('|') if x.strip()}
        return field in parts

    if typ=='CROSS_DOCUMENT_CONFLICT':
        return field in CROSSCHECK_FIELD_MAP.get(str(achamp),set())

    if typ=='INVALID_DATE_ORDER':
        return field in DATE_ORDER_FIELDS.get(str(achamp),set())

    if typ=='DATE_DURATION_INCONSISTENT':
        mapping={
            'DOM_CONTRAT': {'DOM_DUREE_CONTRAT_MOIS','DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT'},
            'TTR_PERMIS': {'TTR_DUREE','TTR_DATE_DEBUT','TTR_DATE_FIN'},
            'CTR_PERMIS_VS_TTR_DUREE': {'TTR_DUREE','CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS'},
            'CTS_PERMIS_VS_TTR_DUREE': {'TTR_DUREE','CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS'},
        }
        return field in mapping.get(str(achamp),set())

    if typ in {'CLASSIFICATION_REVIEW_REQUIRED','EXTRACTION_JSON_VIDE','EXTRACTION_PARTIELLE'}:
        return apage is None or page==apage

    return False


def calculate_dq_summary(processed, field_conf_rows):
    out=_calculate_dq_summary_v55(processed,field_conf_rows)
    records=processed.get('page_records') or []
    page_scores=[
        float(r.get('extraction_confidence_score'))
        for r in records if r.get('extraction_confidence_score') is not None
    ]
    critical_scores=[
        x.get('FIELD_CONFIDENCE') for x in field_conf_rows
        if x.get('DQ_IMPORTANCE')=='CRITICAL' and x.get('FIELD_CONFIDENCE') is not None
    ]
    out['PART1_PAGE_CONFIDENCE_AVG']=round(sum(page_scores)/len(page_scores),1) if page_scores else None
    out['DATA_CONFIDENCE_CRITICAL_AVG']=round(sum(critical_scores)/len(critical_scores),1) if critical_scores else None
    c=out['DATA_CONFIDENCE_CRITICAL_AVG']
    out['DATA_CONFIDENCE_LEVEL']=(
        'HIGH' if c is not None and c>=CONFIDENCE_HIGH else
        'MEDIUM' if c is not None and c>=CONFIDENCE_MEDIUM else
        'LOW' if c is not None and c>0 else 'MISSING'
    )
    out['NB_PAGES_CONFIDENCE_LOW']=sum(r.get('extraction_confidence_band')=='LOW' for r in records)
    return out


def _find_record(processed, page, doc_type):
    for r in processed.get('page_records') or []:
        if r.get('page_num')==page and r.get('doc_type')==doc_type:
            return r
    return None


def _manual_context(processed, row):
    field=row.get('CHAMP')
    rec=_find_record(processed,row.get('PAGE'),row.get('TYPE_DOCUMENT'))
    parts=[]
    if rec:
        parts.append(_field_part1_context(rec,field))
    else:
        missing=_missing_page_context(processed.get('page_records') or [],field)
        if missing:
            parts.append(missing)

    if row.get('NORMALIZATION_STATUS'):
        parts.append(
            f"NORM={row.get('NORMALIZATION_STATUS')}"
            + (f"/{row.get('NORMALIZATION_RULE')}" if row.get('NORMALIZATION_RULE') else '')
        )

    cross=_crosscheck_context_for_field(
        processed.get('page_records') or [],
        processed.get('business_context') or {},
        field
    )
    if cross and cross!='PAS_DE_CROSSCHECK_DEFINI':
        parts.append('COHERENCE=' + cross)

    dur=_duration_context_for_field(processed,field)
    if dur:
        parts.append('DUREE=' + dur)

    if row.get('DQ_ANOMALIES'):
        parts.append('ANOMALIE=' + str(row.get('DQ_ANOMALIES')))
    if row.get('DQ_REASON'):
        parts.append('MOTIF=' + str(row.get('DQ_REASON')))
    return ' | '.join(parts)


def build_manual_validation_rows(processed, field_rows):
    """
    V6 : feuille manuelle compacte.
    Les informations techniques sont concaténées dans CONTEXTE_VALIDATION.
    """
    dq=processed.get('dq_summary') or {}
    ctx=processed.get('business_context') or {}
    conf_map={x.get('CHAMP'):x for x in (processed.get('field_confidence') or [])}
    out=[]

    for r in field_rows:
        if r.get('DQ_STATUS_FIELD') not in {'REVIEW','BLOCKED'}:
            continue
        proposal=propose_best_value(processed,r.get('CHAMP'),r.get('PAGE'),r.get('TYPE_DOCUMENT'))
        conf=conf_map.get(r.get('CHAMP'),{})
        source_bits=[
            x for x in [
                proposal.get('source_document'),
                f"p{proposal.get('source_page')}" if proposal.get('source_page') is not None else None,
                proposal.get('source_field'),
                proposal.get('same_client_status'),
                proposal.get('same_client_evidence'),
                proposal.get('rule'),
            ] if x not in (None,'')
        ]
        out.append({
            'FICHIER':r.get('FICHIER'),
            'TYPE_DOSSIER':ctx.get('TYPE_DOSSIER'),
            'DOSSIER_STATUS':dq.get('VALIDATION_AUTO'),
            'PAGE':r.get('PAGE'),
            'TYPE_DOCUMENT':r.get('TYPE_DOCUMENT'),
            'CHAMP':r.get('CHAMP'),
            'RAW':r.get('RAW'),
            'NORMALIZED':r.get('NORMALIZED'),
            'CONFIDENCE_QUALITE':conf.get('FIELD_CONFIDENCE'),
            'NIVEAU_CONFIDENCE':conf.get('FIELD_CONFIDENCE_LEVEL'),
            'CONTEXTE_VALIDATION':_manual_context(processed,r),
            'VALEUR_PROPOSEE':proposal.get('value'),
            'SOURCE_PROPOSITION':' | '.join(map(str,source_bits)) if source_bits else None,
            'VALEUR_VALIDEE':None,
            'DECISION_MANUELLE':None,
            'COMMENTAIRE':None,
        })

    # Une seule ligne par page cœur absente, au lieu d'une ligne par champ.
    for a in processed.get('anomalies') or []:
        if a.get('TYPE_ANOMALIE')!='EXPECTED_PAGE_ABSENT':
            continue
        out.append({
            'FICHIER':processed.get('source_file'),
            'TYPE_DOSSIER':ctx.get('TYPE_DOSSIER'),
            'DOSSIER_STATUS':dq.get('VALIDATION_AUTO'),
            'PAGE':None,
            'TYPE_DOCUMENT':a.get('TYPE_DOCUMENT'),
            'CHAMP':'[PAGE_ABSENTE]',
            'RAW':None,'NORMALIZED':None,
            'CONFIDENCE_QUALITE':0,'NIVEAU_CONFIDENCE':'MISSING',
            'CONTEXTE_VALIDATION':a.get('MOTIF'),
            'VALEUR_PROPOSEE':None,'SOURCE_PROPOSITION':None,
            'VALEUR_VALIDEE':None,'DECISION_MANUELLE':None,'COMMENTAIRE':None,
        })
    return out


def compact_special_manual_row(row):
    """Compacte les lignes Carthago / compte local dans les mêmes colonnes UX V6."""
    if not row:
        return row
    if 'CONTEXTE_VALIDATION' in row and 'SOURCE_PROPOSITION' in row:
        return row

    context=[]
    for key in [
        'TYPE_ANOMALIE','MOTIF','SIEGE_RACINE','CARTHAGO_CLIENT',
        'CTR_NOM_PRENOM_TRAVAILLEUR','CARTHAGO_NOM_COMPLET',
        'SIMILARITE_NOM','CONTROLE_NOM','CARTHAGO_MATCH_STATUS',
        'PREFIXE_02700_OK','ORDINAL_001_OK','CLE_COMPTE',
        'CLE_ATTENDUE_MOD97','MODULO97_OK','DOM_COMPTE_LOCAL_STATUS'
    ]:
        v=row.get(key)
        if v not in (None,''):
            context.append(f'{key}={v}')

    src=[]
    for key in [
        'SOURCE_PROPOSITION_DOCUMENT','SOURCE_PROPOSITION_PAGE',
        'SOURCE_PROPOSITION_CHAMP','MEME_CLIENT_STATUT',
        'MEME_CLIENT_PREUVE','REGLE_PROPOSITION'
    ]:
        v=row.get(key)
        if v not in (None,''):
            src.append(f'{key}={v}')

    return {
        'FICHIER':row.get('FICHIER'),
        'TYPE_DOSSIER':row.get('TYPE_DOSSIER'),
        'DOSSIER_STATUS':row.get('DOSSIER_STATUS'),
        'PAGE':row.get('PAGE'),
        'TYPE_DOCUMENT':row.get('TYPE_DOCUMENT'),
        'CHAMP':row.get('CHAMP'),
        'RAW':row.get('RAW'),
        'NORMALIZED':row.get('NORMALIZED'),
        'CONFIDENCE_QUALITE':(
            98 if row.get('CHAMP')=='NUMERO_DOMICILIATION_VALIDE'
               and row.get('CARTHAGO_MATCH_STATUS') in {
                   'EXACT_DOMICILIATION_REFERENCE_MATCH',
                   'EXACT_DOMICILIATION_REFERENCE_MATCH_TYPE_UNKNOWN'
               }
            else row.get('FIELD_CONFIDENCE')
        ),
        'NIVEAU_CONFIDENCE':(
            'HIGH' if row.get('CHAMP')=='NUMERO_DOMICILIATION_VALIDE'
               and row.get('CARTHAGO_MATCH_STATUS') in {
                   'EXACT_DOMICILIATION_REFERENCE_MATCH',
                   'EXACT_DOMICILIATION_REFERENCE_MATCH_TYPE_UNKNOWN'
               }
            else None
        ),
        'CONTEXTE_VALIDATION':' | '.join(map(str,context)),
        'VALEUR_PROPOSEE':row.get('VALEUR_PROPOSEE'),
        'SOURCE_PROPOSITION':' | '.join(map(str,src)) if src else None,
        'VALEUR_VALIDEE':row.get('VALEUR_VALIDEE'),
        'DECISION_MANUELLE':row.get('DECISION_MANUELLE'),
        'COMMENTAIRE':row.get('COMMENTAIRE'),
    }


print('✅ Overrides V6 chargés : confidence V9.3.1 + page absente + durée + montants ":" + validation compacte')


## 6. Traitement d’un JSON RAW


In [ ]:

def validate_raw_contract(d):
    return (
        d.get('schema_version')==SCHEMA_VERSION
        and d.get('field_schema_hash')==EXPECTED_FIELD_SCHEMA_HASH
        and d.get('pipeline_version')==EXPECTED_SOURCE_PIPELINE_VERSION
        and isinstance(d.get('page_records'),list)
    )


def process_raw_dossier(d):
    if not validate_raw_contract(d):
        raise ValueError(f"Contrat RAW incompatible : {d.get('source_file')}")

    source=d['source_file']
    processed_records=[]
    anomalies=[]
    retry=[]

    # --------------------------------------------------------------
    # PASSAGE 1 : NORMALISATION UNIQUEMENT
    # --------------------------------------------------------------
    for rec in d['page_records']:
        dt=rec.get('doc_type')
        raw=dict(rec.get('raw_data') or {})
        normalized={}
        trace={}

        for field in FIELD_SCHEMA.get(dt,[]):
            tr=normalize_field_trace(field,raw.get(field))
            trace[field]=tr
            normalized[field]=tr['normalized']

        p=dict(rec)
        p['normalized_data']=normalized
        p['normalization_trace']=trace
        processed_records.append(p)

    # Résolution sécurisée du cas permis TTR affiché droite->gauche.
    rtl_permit_corrections=resolve_ttr_permit_rtl(processed_records)

    business_context=derive_business_context(processed_records)

    # V6 : distinguer explicitement page absente de champ non lu.
    add_expected_page_absence_anomalies(processed_records, anomalies, source)

    # --------------------------------------------------------------
    # PASSAGE 2 : ANOMALIES LIEES AUX PAGES / NORMALISATION
    # --------------------------------------------------------------
    field_rows=[]

    for rec in processed_records:
        dt=rec.get('doc_type')
        page=rec.get('page_num')
        trace=rec.get('normalization_trace') or {}

        flags=page_regulatory_flags(rec,business_context)
        relevant_missing=relevant_critical_missing(rec,business_context)

        if 'CLASSIFICATION_REVIEW_REQUIRED' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'CLASSIFICATION_REVIEW_REQUIRED',
                'SEVERITE':'BLOQUANT','CHAMP':None,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'classification page à revoir'
            })

        if 'EXTRACTION_JSON_VIDE' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'EXTRACTION_JSON_VIDE',
                'SEVERITE':'BLOQUANT','CHAMP':None,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'extraction vide/échouée'
            })

        if 'EXTRACTION_PARTIELLE' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'EXTRACTION_PARTIELLE',
                'SEVERITE':'REVIEW','CHAMP':None,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'extraction partielle'
            })

        if 'CRITICAL_FIELD_MISSING' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'CRITICAL_FIELD_MISSING',
                'SEVERITE':'BLOQUANT',
                'CHAMP':' | '.join(relevant_missing),
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'champ critique réellement absent après réévaluation Partie 2'
            })

        for field in FIELD_SCHEMA.get(dt,[]):
            tr=trace[field]
            applicability,importance=field_policy(field,business_context)

            field_rows.append({
                'FICHIER':source,
                'TYPE_DOSSIER':business_context.get('TYPE_DOSSIER'),
                'PAGE':page,
                'TYPE_DOCUMENT':dt,
                'CHAMP':field,
                'TYPE_CHAMP':FIELD_TYPES[field],
                'APPLICABILITE':applicability,
                'DQ_IMPORTANCE':importance,
                'RAW':tr['raw'],
                'NORMALIZED':tr['normalized'],
                # V3 : nom non ambigu.
                'NORMALIZATION_STATUS':tr['status'],
                'NORMALIZATION_RULE':tr['rule'],
                'NORMALIZATION_CHANGED':tr['changed'],
                'NORMALIZATION_MESSAGE':tr.get('message'),
            })

            if tr['status']=='REVIEW' and importance!='INFO_ONLY':
                format_severity='BLOQUANT' if importance=='CRITICAL' else 'REVIEW'
                anomalies.append({
                    'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                    'TYPE_ANOMALIE':'FORMAT_REVIEW','SEVERITE':format_severity,
                    'CHAMP':field,'VALEUR_RAW':tr['raw'],
                    'VALEUR_NORMALISEE':tr['normalized'],
                    'MOTIF':tr['rule']
                })
                retry.append({
                    'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                    'CHAMP':field,'VALEUR_RAW':tr['raw'],
                    'MOTIF':'FORMAT_AMBIGU','RULE':tr['rule']
                })

    # --------------------------------------------------------------
    # PASSAGE 3 : CROSS-CHECK INTER-DOCUMENTS
    # --------------------------------------------------------------
    by_type=defaultdict(list)
    for r in processed_records:
        by_type[r.get('doc_type')].append(r)

    for group in CROSS_DOCUMENT_GROUPS:
        if not crosscheck_group_applicable(group,business_context):
            continue
        vals=[]
        for dt,field in group['fields'].items():
            for r in by_type.get(dt,[]):
                v=(r.get('normalized_data') or {}).get(field)
                if comparable(v):
                    vals.append((
                        dt,field,r.get('page_num'),v,
                        (r.get('raw_data') or {}).get(field)
                    ))

        if len(vals)>=2:
            base=vals[0][3]
            conflict=any(
                not same_value(group['kind'],base,x[3])
                for x in vals[1:]
            )
            if conflict:
                severity='BLOQUANT' if group['name'] in {
                    'SALAIRE_NET','NUMERO_PERMIS',
                    'DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'
                } else 'REVIEW'

                detail=' | '.join(
                    f'{dt}.{field}@p{p}={v}'
                    for dt,field,p,v,raw in vals
                )

                anomalies.append({
                    'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':'MULTI',
                    'TYPE_ANOMALIE':'CROSS_DOCUMENT_CONFLICT',
                    'SEVERITE':severity,
                    'CHAMP':group['name'],
                    'VALEUR_RAW':None,
                    'VALEUR_NORMALISEE':detail,
                    'MOTIF':'valeurs divergentes entre documents'
                })

                for dt,field,p,v,raw in vals:
                    retry.append({
                        'FICHIER':source,'PAGE':p,'TYPE_DOCUMENT':dt,
                        'CHAMP':field,'VALEUR_RAW':raw,
                        'MOTIF':'CROSS_DOCUMENT_CONFLICT',
                        'RULE':group['name']
                    })

    # --------------------------------------------------------------
    # PASSAGE 4 : CONTROLES METIER + COMPLETUDE CRITIQUE PARTIE 2
    # --------------------------------------------------------------
    temp_processed={
        'source_file':source,
        'page_records':processed_records,
        'business_context':business_context,
    }

    add_business_controls(temp_processed,anomalies,business_context)

    add_postprocess_critical_missing(
        processed_records,anomalies,business_context,source
    )

    # V6 : exploiter les confidences opérationnelles de la Partie 1 V9.3.1.
    add_part1_confidence_anomalies(
        processed_records,anomalies,business_context,source
    )

    anomalies=dedupe_anomalies(anomalies)

    # --------------------------------------------------------------
    # PASSAGE 5 : CONFIDENCE, SCORE DOSSIER, STATUT FINAL DES CHAMPS
    # --------------------------------------------------------------
    field_confidence=calculate_field_confidence(
        processed_records,business_context,anomalies
    )

    out={
        'schema_version':SCHEMA_VERSION,
        'source_file':source,
        'source_sha256':d.get('source_sha256'),
        'source_pipeline_version':d.get('pipeline_version'),
        'postprocess_version':POSTPROCESS_VERSION,
        'normalization_version':NORMALIZATION_VERSION,
        'processed_at':datetime.now().isoformat(timespec='seconds'),
        'business_context':business_context,
        'page_records':processed_records,
        'anomalies':anomalies,
        'retry_requests':retry,
        'field_confidence':field_confidence,
        'source_stats':d.get('stats') or {},
        'rtl_permit_corrections':rtl_permit_corrections,
        'duration_checks':temp_processed.get('duration_checks') or [],
    }

    dq_summary=calculate_dq_summary(out,field_confidence)
    out['dq_summary']=dq_summary

    field_rows=enrich_field_rows(
        field_rows,
        anomalies,
        field_confidence,
        dq_summary.get('VALIDATION_AUTO')
    )

    manual_rows=build_manual_validation_rows(out,field_rows)
    crosscheck_rows=build_crosscheck_rows(out)

    return (
        out,field_rows,anomalies,retry,field_confidence,
        dq_summary,manual_rows,crosscheck_rows
    )


def first_non_null(*values):
    for v in values:
        if v not in (None,''):
            return v
    return None


def consolidate_for_validation(processed):
    records=processed['page_records']
    ctx=processed.get('business_context') or {}

    row={
        'FICHIER':processed['source_file'],
        'TYPE_DOSSIER':ctx.get('TYPE_DOSSIER'),
        'TYPE_DOSSIER_MOTIF':ctx.get('TYPE_DOSSIER_MOTIF'),
        'CTS_DATE_DOCUMENT_AUGMENTATION':ctx.get('CTS_DATE_DOCUMENT_AUGMENTATION'),
        'PERIODE_EFFET_AUGMENTATION':ctx.get('PERIODE_EFFET_AUGMENTATION'),
    }

    row['NB_PAGES']=len(set(
        r.get('page_num') for r in records
        if r.get('page_num') is not None
    ))

    row['TYPES_DOCUMENTS']=' | '.join(dict.fromkeys(
        str(r.get('doc_type')) for r in records
    ))

    for r in records:
        nd=r.get('normalized_data') or {}
        for f,v in nd.items():
            if f not in row or row.get(f) in (None,''):
                row[f]=v

    for f in ALL_FIELDS:
        row.setdefault(f,None)

    row['NOM_TRAVAILLEUR_REFERENCE']=first_non_null(
        row.get('CTR_NOM_PRENOM_TRAVAILLEUR'),
        row.get('CTS_NOM_PRENOM_TRAVAILLEUR'),
        ' '.join(
            x for x in [
                str(row.get('TTR_NOM') or '').strip(),
                str(row.get('TTR_PRENOM') or '').strip()
            ] if x
        ) or None
    )

    row['NUMERO_PERMIS_REFERENCE']=first_non_null(
        row.get('TTR_NUMERO_PERMIS'),
        row.get('CTR_NUMERO_PERMIS_TRAVAIL'),
        row.get('CTS_NUMERO_PERMIS_TRAVAIL')
    )

    row['DATE_DEBUT_CONTRAT_REFERENCE']=first_non_null(
        row.get('CTS_DATE_DEBUT_CONTRAT'),
        row.get('CTR_DATE_DEBUT_CONTRAT'),
        row.get('DOM_DATE_DEBUT_CONTRAT')
    )

    row['DATE_FIN_CONTRAT_REFERENCE']=first_non_null(
        row.get('DOM_DATE_FIN_CONTRAT')
    )

    row['SALAIRE_REFERENCE']=first_non_null(
        row.get('CTS_SALAIRE_NET'),
        row.get('CTR_SALAIRE_NET'),
        row.get('DOM_SALAIRE_NET_MENSUEL')
    )

    row['PART_TRANSFERABLE_REFERENCE']=first_non_null(
        row.get('CTS_PART_TRANSFERABLE'),
        row.get('DOM_PART_TRANSFERABLE')
    )

    # Références vigilance — priorité CTS.
    row['PERE_NOM_PRENOM_REFERENCE']=first_non_null(
        row.get('CTS_PERE_NOM_PRENOM'),
        row.get('CTR_PERE_NOM_PRENOM')
    )
    row['MERE_NOM_PRENOM_REFERENCE']=first_non_null(
        row.get('CTS_MERE_NOM_PRENOM'),
        row.get('CTR_MERE_NOM_PRENOM')
    )
    row['DUREE_CONTRAT_MOIS_REFERENCE']=first_non_null(
        row.get('CTS_DUREE_MOIS'),
        row.get('CTR_DUREE_MOIS'),
        row.get('DOM_DUREE_CONTRAT_MOIS')
    )

    anomalies=processed.get('anomalies') or []
    row['NB_ANOMALIES']=len(anomalies)
    row['NB_BLOQUANTS']=sum(a.get('SEVERITE')=='BLOQUANT' for a in anomalies)
    row['NB_REVIEW']=sum(a.get('SEVERITE')=='REVIEW' for a in anomalies)

    dq=processed.get('dq_summary') or {}
    for k,v in dq.items():
        if k not in {'TYPE_DOSSIER','CTS_DATE_AUGMENTATION','CTS_DATE_DOCUMENT_AUGMENTATION','PERIODE_EFFET_AUGMENTATION'}:
            row[k]=v

    row['NB_RETRY_REQUESTS']=len(processed.get('retry_requests') or [])
    row['STATUT_VALIDATION']=row.get('VALIDATION_AUTO','REVIEW')
    row['COMMENTAIRE_VALIDATION']=None

    return row,{}


## 7. Exécution Partie 2 et classeur de validation


In [ ]:

# Tests de non-régression V6

# Montants avec ':' issus de l'impression DOM.
_t=normalize_amount_trace('454:835:67')
assert _t['status'] in {'RAW_OK','AUTO_OK'} and abs(_t['normalized']-454835.67)<0.001, _t
_t=normalize_amount_trace('47:719:857,6')
assert _t['status'] in {'RAW_OK','AUTO_OK'} and abs(_t['normalized']-47719857.60)<0.001, _t
_t=normalize_amount_trace('23:340.43')
assert _t['status'] in {'RAW_OK','AUTO_OK'} and abs(_t['normalized']-23340.43)<0.001, _t

# Durée TTR : 2 ans cohérents vs cas observé 3 ans.
_ok=_duration_check('TTR_PERMIS','03/01/2026','02/01/2028','2 ANS, 0 JOURS')
assert _ok and _ok['ok'] is True, _ok
_bad=_duration_check('TTR_PERMIS','02/01/2023','03/01/2026','2 ANS, 0 JOURS')
assert _bad and _bad['ok'] is False, _bad

# Une confidence P1 82 ne doit pas être assimilée automatiquement à une erreur.
assert CONFIDENCE_CRITICAL_REVIEW < 82

print('✅ Tests V6 : montants ":" + durée + seuil confidence OK')


In [ ]:


# ---------------------------------------------------------------------
# V5.3 — Chargement CarthagoDom
# ---------------------------------------------------------------------
CARTHAGO_DF, CARTHAGO_META = load_carthago_reference(CARTHAGO_XLSX)
print('CarthagoDom :', CARTHAGO_META.get('message'))
print('  Client    :', CARTHAGO_META.get('client_col'))
print('  Nom       :', CARTHAGO_META.get('name_col'))
print('  Num DOM   :', CARTHAGO_META.get('dom_ref_col'))
print('  Type      :', CARTHAGO_META.get('type_col'))
if CARTHAGO_META.get('available'):
    print('  Lignes    :', len(CARTHAGO_DF))

json_files=sorted(RAW_JSON_DIR.glob('*.json'))
if MAX_DOSSIERS is not None:
    json_files=json_files[:int(MAX_DOSSIERS)]
print('JSON RAW sélectionnés :',len(json_files))

all_dossiers=[]
all_fields=[]
all_anomalies=[]
all_retry=[]
all_docs=[]
all_field_confidence=[]
all_dq_summary=[]
all_manual=[]
all_crosschecks=[]
all_carthago_matches=[]
errors=[]

for i,p in enumerate(json_files,1):
    try:
        raw=json.loads(p.read_text(encoding='utf-8'))

        (
            processed,field_rows,anomalies,retry,
            field_confidence,dq_summary,
            manual_rows,crosscheck_rows
        )=process_raw_dossier(raw)

        # V5.3 — rapprochement Carthago, validation du numéro DOM et contrôle du compte local.
        carthago_summary, carthago_details = match_domiciliation_carthago(
            processed, CARTHAGO_DF, CARTHAGO_META
        )
        processed['domiciliation_validation'] = carthago_summary
        manual_rows.append(
            compact_special_manual_row(
                build_domiciliation_manual_row(processed, carthago_summary, CARTHAGO_META)
            )
        )
        compte_manual_row = build_compte_local_manual_row(processed, carthago_summary)
        if compte_manual_row is not None:
            manual_rows.append(compact_special_manual_row(compte_manual_row))
        all_carthago_matches.extend(carthago_details)

        row,_=consolidate_for_validation(processed)
        row.update({
            'DOM_COMPTE_LOCAL_NORMALIZED': carthago_summary.get('DOM_COMPTE_LOCAL_NORMALIZED'),
            'SIEGE_RACINE': carthago_summary.get('SIEGE_RACINE'),
            'RACINE_6': carthago_summary.get('RACINE_6'),
            'PREFIXE_02700_OK': carthago_summary.get('PREFIXE_02700_OK'),
            'ORDINAL_001_OK': carthago_summary.get('ORDINAL_001_OK'),
            'CLE_COMPTE': carthago_summary.get('CLE_COMPTE'),
            'CLE_ATTENDUE_MOD97': carthago_summary.get('CLE_ATTENDUE_MOD97'),
            'MODULO97_RESTE': carthago_summary.get('MODULO97_RESTE'),
            'MODULO97_OK': carthago_summary.get('MODULO97_OK'),
            'DOM_COMPTE_LOCAL_STATUS': carthago_summary.get('DOM_COMPTE_LOCAL_STATUS'),
            'CARTHAGO_CLIENT_MATCH': carthago_summary.get('CARTHAGO_CLIENT_MATCH'),
            'NUMERO_DOMICILIATION_OCR': carthago_summary.get('CTR_REFERENCE_DOMICILIATION_NORMALIZED'),
            'NUMERO_DOMICILIATION_NORMALIZED': carthago_summary.get('NUMERO_DOMICILIATION_NORMALIZED'),
            'NUMERO_DOMICILIATION_PROPOSE': carthago_summary.get('NUMERO_DOMICILIATION_PROPOSE'),
            'CARTHAGO_MATCH_STATUS': carthago_summary.get('CARTHAGO_MATCH_STATUS'),
            'SIMILARITE_NOM_CARTHAGO': carthago_summary.get('SIMILARITE_NOM'),
        })

        all_dossiers.append(row)
        all_fields.extend(field_rows)
        all_anomalies.extend(anomalies)
        all_retry.extend(retry)
        all_manual.extend(manual_rows)
        all_crosschecks.extend(crosscheck_rows)

        all_field_confidence.extend([
            {'FICHIER':processed['source_file'], **x}
            for x in field_confidence
        ])

        all_dq_summary.append({
            'FICHIER':processed['source_file'],
            'TYPE_DOSSIER':(processed.get('business_context') or {}).get('TYPE_DOSSIER'),
            'CTS_DATE_DOCUMENT_AUGMENTATION':(processed.get('business_context') or {}).get('CTS_DATE_DOCUMENT_AUGMENTATION'),
            **dq_summary,
            'NB_ANOMALIES':len(anomalies)
        })

        for r in processed['page_records']:
            all_docs.append({
                'FICHIER':processed['source_file'],
                'PAGE':r.get('page_num'),
                'TYPE_DOCUMENT':r.get('doc_type'),
                'STATUT_EXTRACTION':r.get('extraction_status'),
                'TAUX_REMPLISSAGE':r.get('extraction_taux_remplissage'),
                'CLASSIFICATION_REVIEW_REQUIRED':r.get('classification_review_required'),
                'CRITICAL_FIELDS_MISSING_RAW':' | '.join(r.get('critical_fields_missing') or []),
                'CRITICAL_FIELDS_MISSING_DQ':' | '.join(
                    relevant_critical_missing(
                        r,processed.get('business_context') or {}
                    )
                ),
                'QUALITY_FLAGS':' | '.join(r.get('quality_flags') or []),
                'P1_PAGE_CONFIDENCE':r.get('extraction_confidence_score'),
                'P1_PAGE_CONFIDENCE_BAND':r.get('extraction_confidence_band'),
                'P1_CONFIDENCE_METHOD':r.get('confidence_method'),
                'PAGE_RECOVERY_TRIGGERED':r.get('page_recovery_triggered'),
                'RECOVERY_PASSES':r.get('recovery_passes'),
                'FIELD_DISAGREEMENTS':len(r.get('field_disagreements') or []),
                'FIELD_REVISIONS':len(r.get('field_revisions') or []),
                'STRATEGIES':' > '.join(
                    s.get('nom','')
                    for s in (r.get('extraction_strategies') or [])
                ),
                'TOKENS_IN':r.get('extraction_tokens_in'),
                'TOKENS_OUT':r.get('extraction_tokens_out')
            })

        outpath=PROCESSED_JSON_DIR/p.name
        outpath.write_text(
            json.dumps(processed,ensure_ascii=False,indent=2,default=str),
            encoding='utf-8'
        )

        print(
            f'[{i}/{len(json_files)}] ✅ {p.name} | '
            f"status={dq_summary.get('VALIDATION_AUTO')} | "
            f"anomalies={len(anomalies)}"
        )

    except Exception as exc:
        errors.append({'FICHIER':p.name,'ERREUR':repr(exc)})
        print(f'[{i}/{len(json_files)}] ❌ {p.name}: {exc}')


DF_DOSSIERS=pd.DataFrame(all_dossiers)
DF_FIELDS=pd.DataFrame(all_fields)
DF_ANOMALIES=pd.DataFrame(all_anomalies)
DF_RETRY=(
    pd.DataFrame(all_retry).drop_duplicates()
    if all_retry else
    pd.DataFrame(columns=[
        'FICHIER','PAGE','TYPE_DOCUMENT','CHAMP',
        'VALEUR_RAW','MOTIF','RULE'
    ])
)
DF_DOCS=pd.DataFrame(all_docs)
DF_ERRORS=pd.DataFrame(errors)
DF_FIELD_CONFIDENCE=pd.DataFrame(all_field_confidence)
DF_DQ_SUMMARY=pd.DataFrame(all_dq_summary)
DF_MANUAL=pd.DataFrame(all_manual)
MANUAL_COLUMNS=[
    'FICHIER','TYPE_DOSSIER','DOSSIER_STATUS','PAGE','TYPE_DOCUMENT','CHAMP',
    'RAW','NORMALIZED','CONFIDENCE_QUALITE','NIVEAU_CONFIDENCE',
    'CONTEXTE_VALIDATION','VALEUR_PROPOSEE','SOURCE_PROPOSITION',
    'VALEUR_VALIDEE','DECISION_MANUELLE','COMMENTAIRE'
]
for _c in MANUAL_COLUMNS:
    if _c not in DF_MANUAL.columns:
        DF_MANUAL[_c]=None
DF_MANUAL=DF_MANUAL[MANUAL_COLUMNS]
DF_CROSSCHECK=pd.DataFrame(all_crosschecks)
DF_CARTHAGO_MATCH=pd.DataFrame(all_carthago_matches)

DF_DOSSIERS.to_csv(DOSSIERS_CSV,index=False,encoding='utf-8-sig')
DF_FIELDS.to_csv(FIELDS_CSV,index=False,encoding='utf-8-sig')
DF_ANOMALIES.to_csv(ANOMALIES_CSV,index=False,encoding='utf-8-sig')
DF_RETRY.to_csv(RETRY_CSV,index=False,encoding='utf-8-sig')
DF_FIELD_CONFIDENCE.to_csv(FIELD_CONFIDENCE_CSV,index=False,encoding='utf-8-sig')
DF_DQ_SUMMARY.to_csv(DQ_SUMMARY_CSV,index=False,encoding='utf-8-sig')
DF_CARTHAGO_MATCH.to_csv(CARTHAGO_MATCH_CSV,index=False,encoding='utf-8-sig')

with pd.ExcelWriter(VALIDATION_XLSX,engine='openpyxl') as writer:
    # Ordre des feuilles : opérationnel d'abord.
    DF_DQ_SUMMARY.to_excel(writer,sheet_name='DQ_DASHBOARD',index=False)
    DF_MANUAL.to_excel(writer,sheet_name='A_VALIDER_MANUELLEMENT',index=False)
    DF_DOSSIERS.to_excel(writer,sheet_name='DOSSIERS_A_VALIDER',index=False)
    DF_CROSSCHECK.to_excel(writer,sheet_name='CONTROLES_COHERENCE',index=False)
    DF_CARTHAGO_MATCH.to_excel(writer,sheet_name='CARTHAGO_MATCH',index=False)
    DF_FIELDS.to_excel(writer,sheet_name='CHAMPS_DETAIL',index=False)
    DF_ANOMALIES.to_excel(writer,sheet_name='ANOMALIES',index=False)
    DF_FIELD_CONFIDENCE.to_excel(writer,sheet_name='FIELD_CONFIDENCE',index=False)
    DF_RETRY.to_excel(writer,sheet_name='VLM_RETRY_REQUESTS',index=False)
    DF_DOCS.to_excel(writer,sheet_name='DOCUMENTS',index=False)
    DF_ERRORS.to_excel(writer,sheet_name='ERREURS',index=False)

    pd.DataFrame([
        {'PARAMETRE':'schema_version','VALEUR':SCHEMA_VERSION},
        {'PARAMETRE':'field_schema_hash','VALEUR':EXPECTED_FIELD_SCHEMA_HASH},
        {'PARAMETRE':'postprocess_version','VALEUR':POSTPROCESS_VERSION},
        {'PARAMETRE':'normalization_version','VALEUR':NORMALIZATION_VERSION},
        {'PARAMETRE':'normalization_status_note','VALEUR':'NORMALIZATION_STATUS concerne uniquement le format de la cellule'},
        {'PARAMETRE':'field_status_note','VALEUR':'DQ_STATUS_FIELD intègre normalisation + extraction + cross-check + contrôles métier'},
        {'PARAMETRE':'dossier_status_note','VALEUR':'AUTO_OK uniquement si aucune anomalie BLOQUANT/REVIEW et DQ_SCORE >= 90'},
        {'PARAMETRE':'dq_score_note','VALEUR':'Indice interne explicable 0-100; ce n est pas une probabilité ni un taux de confiance Qwen'},
        {'PARAMETRE':'confidence_v6','VALEUR':'DATA_CONFIDENCE 0-100 = confidence opérationnelle V9.3.1 + normalisation + cross-check + cohérence métier; ce n est pas une probabilité native Qwen'},
        {'PARAMETRE':'type_dossier_rule','VALEUR':'AUGMENTATION si CTS_SALAIRE_NET_ANCIEN présent ou CTS_MENTION_AU_LIEU_DE_PRESENTE=True; sinon NOUVEAU_CONTRAT si CTS présent'},
        {'PARAMETRE':'augmentation_effect_rule','VALEUR':'V6: la date CTS/document n est PAS la date d effet; le planning utilisera un fichier annuel ANNEE_AUGMENTATION + MOIS_AUGMENTATION'},
        {'PARAMETRE':'manual_proposal_priority','VALEUR':'CONTRAT_SPECIFIQUE > CONTRAT_TRAVAIL > TITRE_TRAVAIL > ENGAGEMENT_DOMICILIATION'},
        {'PARAMETRE':'manual_proposal_same_client','VALEUR':'Proposition uniquement si le document source est confirmé comme même client'},
        {'PARAMETRE':'permit_rtl_rule','VALEUR':'TTR permis: inversion ordre des groupes uniquement si match CTR/CTS + identité nom/date naissance'},
        {'PARAMETRE':'ctr_dom_ref_rule','VALEUR':'Format structuré canonique PREFIXE|AAAA.T|40|SEQUENCE|DZD; 27-19-01->271901; période sans point acceptée; devise vide/D/DZ complétée en DZD; aucune conversion OCR lettre->chiffre'},
        {'PARAMETRE':'numero_dom_compare_rule','VALEUR':'NUMERO_DOMICILIATION_VALIDE / comparaison Carthago = tous les chiffres de la référence, sans séparateurs ni lettres, puis suffixe DZD'},
        {'PARAMETRE':'siege_racine_rule','VALEUR':'Regex 07310 + 6 chiffres depuis DOM_COMPTE_LOCAL; comparaison exacte avec Client Carthago; aucune découpe par position fixe'},
        {'PARAMETRE':'dom_compte_local_rule','VALEUR':'Contrôles Tosyali séparés: 20 chiffres, préfixe 02700, regex 07310+6 chiffres, ordinal 001 immédiatement après; aucune correction automatique'},
        {'PARAMETRE':'dom_compte_local_mod97','VALEUR':'V6: RIB 20 chiffres = banque(3)+agence(5)+compte(10)+clé(2). Vérification modulo97 sur agence+compte+clé (code banque exclu) et calcul de la clé attendue sur agence+compte; aucune correction automatique'},
        {'PARAMETRE':'carthago_match_rule','VALEUR':'SIEGE_RACINE == Client -> similarité nom informationnelle -> comparaison exacte NUMERO_DOMICILIATION_VALIDE normalisé avec Numero de domiciliation Carthago normalisé; PREDOM seul non certifié'},
        {'PARAMETRE':'missing_page_rule','VALEUR':'Une page coeur absente est signalée une seule fois dans A_VALIDER_MANUELLEMENT avec la liste des champs critiques indisponibles; pas une ligne par champ'},
        {'PARAMETRE':'amount_colon_rule','VALEUR':"Sur les montants, ':' est interprété comme séparateur d impression équivalent à '.' uniquement pour une chaîne numérique structurée"},
        {'PARAMETRE':'duration_rule','VALEUR':'Contrôle dates début/fin vs durée pour DOM et TTR; TTR_DUREE confirme aussi les dates de permis CTR/CTS lorsqu elles sont disponibles'},
        {'PARAMETRE':'manual_sheet_rule','VALEUR':'Colonnes techniques regroupées dans CONTEXTE_VALIDATION et SOURCE_PROPOSITION pour réduire le nombre de colonnes'}, 
    ]).to_excel(writer,sheet_name='PARAMETRES',index=False)

    # Format monétaire 2 décimales dans le consolidé.
    ws=writer.book['DOSSIERS_A_VALIDER']
    amount_cols=set(AMOUNT_FIELDS) | {
        'SALAIRE_REFERENCE','PART_TRANSFERABLE_REFERENCE'
    }
    headers={cell.value:cell.column for cell in ws[1]}
    for col_name in amount_cols:
        col_idx=headers.get(col_name)
        if col_idx:
            for row_idx in range(2,ws.max_row+1):
                cell=ws.cell(row=row_idx,column=col_idx)
                if isinstance(cell.value,(int,float)):
                    cell.number_format='0.00'

    # Format montant CHAMPS_DETAIL.
    ws=writer.book['CHAMPS_DETAIL']
    headers={cell.value:cell.column for cell in ws[1]}
    type_col=headers.get('TYPE_CHAMP')
    norm_col=headers.get('NORMALIZED')
    if type_col and norm_col:
        for row_idx in range(2,ws.max_row+1):
            if ws.cell(row=row_idx,column=type_col).value=='amount':
                cell=ws.cell(row=row_idx,column=norm_col)
                if isinstance(cell.value,(int,float)):
                    cell.number_format='0.00'


# Liste déroulante DECISION_MANUELLE.
if 'A_VALIDER_MANUELLEMENT' in writer.book.sheetnames:
    from openpyxl.worksheet.datavalidation import DataValidation

    ws_manual = writer.book['A_VALIDER_MANUELLEMENT']
    manual_headers = {
        cell.value: cell.column
        for cell in ws_manual[1]
    }
    decision_col = manual_headers.get('DECISION_MANUELLE')

    if decision_col and ws_manual.max_row >= 2:
        dv = DataValidation(
            type='list',
            formula1='"VALIDEE,CORRIGEE,REJETEE"',
            allow_blank=True
        )
        dv.error = 'Choisir VALIDEE, CORRIGEE ou REJETEE.'
        dv.errorTitle = 'Décision invalide'
        dv.prompt = 'Sélectionner la décision de validation.'
        dv.promptTitle = 'Décision manuelle'
        ws_manual.add_data_validation(dv)

        start_cell = ws_manual.cell(row=2, column=decision_col).coordinate
        end_cell = ws_manual.cell(row=ws_manual.max_row, column=decision_col).coordinate
        dv.add(f'{start_cell}:{end_cell}')

    # Figer la première ligne sur les feuilles les plus utilisées.
    for sheet_name in [
        'DQ_DASHBOARD','A_VALIDER_MANUELLEMENT',
        'DOSSIERS_A_VALIDER','CONTROLES_COHERENCE',
        'CHAMPS_DETAIL','ANOMALIES','CARTHAGO_MATCH'
    ]:
        writer.book[sheet_name].freeze_panes='A2'

    # Les validations/freeze panes sont ajoutés après la fermeture du writer pandas.
    # Sauvegarde explicite pour garantir leur persistance dans le fichier final.
    writer.book.save(VALIDATION_XLSX)

print('\n✅ Classeur validation :',VALIDATION_XLSX)
print('✅ Dossiers CSV       :',DOSSIERS_CSV)
print('✅ Champs détail      :',FIELDS_CSV)
print('✅ Anomalies          :',ANOMALIES_CSV)
print('✅ Retry requests     :',RETRY_CSV)
print('✅ DQ summary         :',DQ_SUMMARY_CSV)
print('✅ Field confidence   :',FIELD_CONFIDENCE_CSV)
print('✅ Carthago match      :',CARTHAGO_MATCH_CSV)
print('✅ Nouvelle feuille   : A_VALIDER_MANUELLEMENT')
print('✅ Nouvelle feuille   : CONTROLES_COHERENCE')
print('✅ Nouvelle feuille   : CARTHAGO_MATCH')
print('✅ Validation DOM     : NUMERO_DOMICILIATION_VALIDE compact (chiffres + DZD) ajouté à A_VALIDER_MANUELLEMENT')
print('✅ Contrôle compte     : SIEGE_RACINE + Client + similarité nom + structure Tosyali + Modulo 97')


## Étape suivante après validation V6

Traiter uniquement la feuille `A_VALIDER_MANUELLEMENT`. `CONTEXTE_VALIDATION` regroupe confidence, origine de cohérence, recovery, normalisation et motif. Après validation : Partie 2B pour appliquer `VALIDEE / CORRIGEE / REJETEE`, recalculer les contrôles, puis produire le jeu de données certifié avant `DOM_VERSIONS` et `PLANNING_TL`.
